# Torre de Londres · Busca irrevogável

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/01_busca_irrevogavel.ipynb)

DCC014 Inteligência Artificial · UFJF · Trabalho 1 · Grupo 5

Páginas: **Busca irrevogável** · [Backtracking](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/02_backtracking.ipynb) · [Busca em largura](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/03_busca_em_largura.ipynb) · [Busca ordenada](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/04_busca_ordenada.ipynb) · [Gráficos e análise](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/05_analise.ipynb)

# Setup

Rode esta seção inteira antes do resto da página.

In [ ]:
import sys
from pathlib import Path

for folder in ('config', 'core/rules/domain', 'core/rules', 'core/rules/strategies/domain', 'core/rules/strategies', 'core/domain', 'core/search_tree', 'core/algorithms/domain', 'core/algorithms', 'libs/inputs', 'libs/outputs', 'libs/ranking', 'runner'):
    Path("src", folder).mkdir(parents=True, exist_ok=True)
if str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))

## config/

In [ ]:
%%writefile src/config/logging.py
from __future__ import annotations

import logging
import sys
from typing import Final

LOGGER_NAME: Final = "torre_de_londres"
LOG_FORMAT: Final = "%(levelname)s %(name)s: %(message)s"


def configure_logging(*, verbose: bool = False) -> logging.Logger:
    logger = logging.getLogger(LOGGER_NAME)
    logger.setLevel(logging.DEBUG if verbose else logging.WARNING)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stderr)
        handler.setFormatter(logging.Formatter(LOG_FORMAT))
        logger.addHandler(handler)
    return logger


def get_logger(suffix: str) -> logging.Logger:
    return logging.getLogger(f"{LOGGER_NAME}.{suffix}")

In [ ]:
%%writefile src/config/settings.py
from __future__ import annotations

from collections.abc import Mapping
from typing import Final

MAX_ITERATIONS: Final = 10_000

DEFAULT_OUTPUT_FORMAT: Final = "console"
DEFAULT_RANK_MODE: Final = "lexicographic"

OUTPUT_FORMATS: Final[tuple[str, ...]] = ("console", "json")
RANK_MODES: Final[tuple[str, ...]] = ("lexicographic", "score")

EXIT_OK: Final = 0
EXIT_INVALID_INPUT: Final = 2


PROGRAM_NAME: Final = "main.py"

DESCRIPTION: Final = (
    "Resolvedor da Torre de Londres: executa busca irrevogável, backtracking, "
    "busca em largura e busca ordenada sobre as cartas do problema, variando a "
    "estratégia de controle, e ranqueia os resultados."
)

EPILOG: Final = (
    'Cada eixo omitido significa "todos": sem --problem roda todas as cartas, '
    "sem --algorithm roda todos os algoritmos, sem --strategy roda todas as "
    "estratégias."
)

HELP_HELP: Final = "Mostra esta ajuda e encerra."
HELP_PROBLEM: Final = "Cartas a resolver (repetível ou separado por vírgula)."
HELP_ALGORITHM: Final = "Algoritmos a executar (repetível ou separado por vírgula)."
HELP_STRATEGY: Final = "Estratégias de controle (repetível ou separado por vírgula)."
HELP_ALL: Final = (
    "Produto cartesiano completo: todas as cartas, algoritmos e estratégias."
)
HELP_ALL_GOALS: Final = (
    "Resolve todos os 36 estados finais possíveis a partir da posição inicial "
    "e mostra só o resumo, com média e mediana de cada algoritmo e estratégia."
)
HELP_ORDER: Final = "Ordem das regras para a estratégia custom, ex.: R2,R4,R6,R1,R3,R5."
HELP_SHOW_TREE: Final = "Renderiza a árvore de busca de cada execução."
HELP_SHOW_TRACE: Final = (
    "Mostra o passo a passo com regra aplicada e estado resultante."
)
HELP_SHOW_STATES: Final = "Desenha as hastes em cada estado do caminho solução."
HELP_FORMAT: Final = "Formato da saída."
HELP_OUTPUT: Final = "Arquivo de destino da saída; sem ele, escreve no terminal."
HELP_SVG_DIR: Final = (
    "Pasta onde gravar a árvore de busca de cada execução em DOT e SVG "
    "(o SVG exige o Graphviz instalado)."
)
HELP_RANK_MODE: Final = "Critério do placar."
HELP_MAX_ITERATIONS: Final = "Guarda contra execução patológica."
HELP_SEED: Final = "Reservado; a execução é determinística por construção."
HELP_LIST: Final = "Lista cartas, algoritmos e estratégias disponíveis e encerra."
HELP_VERBOSE: Final = "Habilita log de diagnóstico na saída de erro."

KIND_LABELS: Final[Mapping[str, str]] = {
    "problem": "Carta não reconhecida",
    "algorithm": "Algoritmo não reconhecido",
    "strategy": "Estratégia não reconhecida",
    "format": "Formato de saída não reconhecido",
    "rank_mode": "Modo de ranqueamento não reconhecido",
}

ERROR_PREFIX: Final = "Erro:"
ERROR_UNKNOWN_SELECTION: Final = "{kind}: {name}. Disponíveis: {available}."
ERROR_INVALID_ARGUMENT: Final = "argumento inválido: {detail}."
ERROR_INVALID_RULE_ORDER: Final = "ordem de regras inválida: {reason}."
ERROR_HINT: Final = "Use --help para ver as opções."

USAGE_PREFIX: Final = "Uso: "
OPTIONS_TITLE: Final = "Opções"

LIST_PROBLEMS_HEADER: Final = "Cartas disponíveis"
LIST_ALGORITHMS_HEADER: Final = "Algoritmos disponíveis"
LIST_STRATEGIES_HEADER: Final = "Estratégias disponíveis"
OUTPUT_WRITTEN: Final = "Saída gravada em {path}"
GRAPHS_WRITTEN: Final = "{count} arquivos de árvore gravados em {path}"
GRAPHVIZ_MISSING: Final = (
    "Aviso: Graphviz (dot) não encontrado; só os arquivos .dot foram gravados."
)

## core/rules/domain/

In [ ]:
%%writefile src/core/rules/domain/base.py
"""Legenda do problema.

Discos: VERDE, VERMELHO, AZUL. Distintos por cor; NÃO há ordenação por tamanho.
Hastes: H1, H2, H3.

Capacidades:
    H1: até 3 discos
    H2: até 2 discos
    H3: até 1 disco

A capacidade é a única restrição estrutural: é o que substitui a ordenação por
tamanho da Torre de Hanói.

Estado: tupla de três pilhas, cada uma da base para o topo. Imutável, hashable e
canônica: uma configuração física tem exatamente uma escrita.
"""

from __future__ import annotations

from collections.abc import Mapping
from enum import IntEnum, unique
from types import MappingProxyType
from typing import Protocol


@unique
class Disk(IntEnum):
    GREEN = 0
    RED = 1
    BLUE = 2


@unique
class Peg(IntEnum):
    H1 = 0
    H2 = 1
    H3 = 2


CAPACITIES: Mapping[Peg, int] = MappingProxyType({Peg.H1: 3, Peg.H2: 2, Peg.H3: 1})

TOTAL_DISKS: int = len(Disk)

type Stack = tuple[Disk, ...]
type State = tuple[Stack, Stack, Stack]


class TransitionRule(Protocol):
    """Um movimento: tira o disco do topo de uma haste e põe no topo de outra.

        is_applicable(estado)  ->  posso mover agora? (origem tem disco e
                                   destino tem espaço)
        apply(estado)          ->  o estado que resulta do movimento
        cost                   ->  quanto custa mover (só a busca ordenada usa)

    `apply` não mexe no estado recebido, devolve um novo -- por isso um nó da
    árvore nunca perde a configuração que guardava. São seis regras fixas, R1
    a R6, uma por par origem/destino, definidas em `moves.py`.

    A busca não sabe o que é disco nem haste: ela só testa a regra, aplica a
    que passou, e segue. Trocar o problema é trocar as regras, não o motor.
    """

    @property
    def id(self) -> str: ...

    @property
    def origin(self) -> Peg: ...

    @property
    def destination(self) -> Peg: ...

    @property
    def cost(self) -> int: ...

    def is_applicable(self, state: State) -> bool: ...

    def apply(self, state: State) -> State: ...

In [ ]:
%%writefile src/core/rules/domain/catalog.py
from __future__ import annotations

from collections.abc import Mapping
from types import MappingProxyType

from core.rules.domain.base import TransitionRule
from core.rules.domain.exceptions import UnknownRuleError
from core.rules.moves import R1, R2, R3, R4, R5, R6

RULES: tuple[TransitionRule, ...] = (R1, R2, R3, R4, R5, R6)

RULE_BY_ID: Mapping[str, TransitionRule] = MappingProxyType(
    {rule.id: rule for rule in RULES}
)

INVERSE_RULE_ID: Mapping[str, str] = MappingProxyType(
    {"R1": "R3", "R3": "R1", "R2": "R5", "R5": "R2", "R4": "R6", "R6": "R4"}
)


def get_rule(rule_id: str) -> TransitionRule:
    try:
        return RULE_BY_ID[rule_id]
    except KeyError:
        raise UnknownRuleError(rule_id) from None


def inverse_of(rule: TransitionRule) -> TransitionRule:
    return RULE_BY_ID[INVERSE_RULE_ID[rule.id]]

In [ ]:
%%writefile src/core/rules/domain/constraints.py
"""Condições de validade.

    1. Origem: a haste de origem possui pelo menos um disco no momento exato da
       ação.
    2. Destino: a haste de destino não ultrapassa sua capacidade estrita de
       volume (H1 até 3, H2 até 2, H3 até 1).

As duas são estáticas: dependem apenas do estado corrente.

A terceira condição do material, que proíbe produzir estado já presente no
caminho, é contextual: depende do percurso, e por isso vive no algoritmo
(`core/search_tree/path.py`), não aqui.
"""

from __future__ import annotations

from core.rules.domain.base import CAPACITIES, Peg, State


def has_disk_to_move(state: State, origin: Peg) -> bool:
    return len(state[origin]) >= 1


def has_room_for_disk(state: State, destination: Peg) -> bool:
    return len(state[destination]) + 1 <= CAPACITIES[destination]


def is_move_allowed(state: State, origin: Peg, destination: Peg) -> bool:
    return has_disk_to_move(state, origin) and has_room_for_disk(state, destination)

In [ ]:
%%writefile src/core/rules/domain/exceptions.py
class RuleError(Exception): ...


class RuleNotApplicableError(RuleError):
    def __init__(self, rule_id: str) -> None:
        super().__init__(f"rule {rule_id} is not applicable to the given state")
        self.rule_id = rule_id


class UnknownRuleError(RuleError):
    def __init__(self, rule_id: str) -> None:
        super().__init__(f"unknown rule identifier: {rule_id!r}")
        self.rule_id = rule_id


class InvalidRuleOrderError(RuleError):
    def __init__(self, reason: str) -> None:
        super().__init__(f"invalid rule order: {reason}")
        self.reason = reason


class UnknownStrategyError(RuleError):
    def __init__(self, name: str) -> None:
        super().__init__(f"unknown control strategy: {name!r}")
        self.name = name

## core/rules/

In [ ]:
%%writefile src/core/rules/moves.py
"""Regras de transição de estado.

    R1: mover o disco do topo da H1 para a H2.
    R2: mover o disco do topo da H1 para a H3.
    R3: mover o disco do topo da H2 para a H1.
    R4: mover o disco do topo da H2 para a H3.
    R5: mover o disco do topo da H3 para a H1.
    R6: mover o disco do topo da H3 para a H2.

Toda regra tem inversa exata: R1<->R3, R2<->R5, R4<->R6.

Custo de uma regra: a distância entre as hastes que ela liga, como um braço
que leva o disco de uma haste à outra.

    R1, R3 (H1 <-> H2): 1     R4, R6 (H2 <-> H3): 1     R2, R5 (H1 <-> H3): 2

Uma regra e sua inversa custam o mesmo. Só a busca ordenada olha o custo.

`apply` exige `is_applicable` como pré-condição: aplicar regra inválida é erro de
programação e falha alto.
"""

from __future__ import annotations

from dataclasses import dataclass

from core.rules.domain.base import Peg, State
from core.rules.domain.constraints import is_move_allowed
from core.rules.domain.exceptions import RuleNotApplicableError


@dataclass(frozen=True, slots=True)
class Move:
    id: str
    origin: Peg
    destination: Peg

    @property
    def cost(self) -> int:
        return abs(self.destination - self.origin)

    def is_applicable(self, state: State) -> bool:
        return is_move_allowed(state, self.origin, self.destination)

    def apply(self, state: State) -> State:
        if not self.is_applicable(state):
            raise RuleNotApplicableError(self.id)

        stacks = list(state)
        disk = stacks[self.origin][-1]
        stacks[self.origin] = stacks[self.origin][:-1]
        stacks[self.destination] = (*stacks[self.destination], disk)
        return (stacks[0], stacks[1], stacks[2])


R1 = Move("R1", Peg.H1, Peg.H2)
R2 = Move("R2", Peg.H1, Peg.H3)
R3 = Move("R3", Peg.H2, Peg.H1)
R4 = Move("R4", Peg.H2, Peg.H3)
R5 = Move("R5", Peg.H3, Peg.H1)
R6 = Move("R6", Peg.H3, Peg.H2)

## core/rules/strategies/domain/

In [ ]:
%%writefile src/core/rules/strategies/domain/base.py
from __future__ import annotations

from typing import Protocol

from core.rules.domain.base import TransitionRule


class ControlStrategy(Protocol):
    @property
    def name(self) -> str: ...

    def order(
        self, rules: tuple[TransitionRule, ...]
    ) -> tuple[TransitionRule, ...]: ...

In [ ]:
%%writefile src/core/rules/strategies/domain/registry.py
from __future__ import annotations

from collections.abc import Mapping
from types import MappingProxyType

from core.rules.domain.exceptions import UnknownStrategyError
from core.rules.strategies.ascending import ASCENDING
from core.rules.strategies.custom_order import CUSTOM
from core.rules.strategies.descending import DESCENDING
from core.rules.strategies.domain.base import ControlStrategy

STRATEGIES: Mapping[str, ControlStrategy] = MappingProxyType(
    {
        ASCENDING.name: ASCENDING,
        DESCENDING.name: DESCENDING,
        CUSTOM.name: CUSTOM,
    }
)

STRATEGY_NAMES: tuple[str, ...] = tuple(STRATEGIES)


def get_strategy(name: str) -> ControlStrategy:
    try:
        return STRATEGIES[name]
    except KeyError:
        raise UnknownStrategyError(name) from None

## core/rules/strategies/

In [ ]:
%%writefile src/core/rules/strategies/ascending.py
from __future__ import annotations

from dataclasses import dataclass

from core.rules.domain.base import TransitionRule
from core.rules.domain.catalog import RULES

_CANONICAL_POSITION = {rule.id: position for position, rule in enumerate(RULES)}


@dataclass(frozen=True, slots=True)
class AscendingStrategy:
    name: str = "ascending"

    def order(self, rules: tuple[TransitionRule, ...]) -> tuple[TransitionRule, ...]:
        return tuple(sorted(rules, key=lambda rule: _CANONICAL_POSITION[rule.id]))


ASCENDING = AscendingStrategy()

In [ ]:
%%writefile src/core/rules/strategies/custom_order.py
from __future__ import annotations

from dataclasses import dataclass, field

from core.rules.domain.base import TransitionRule
from core.rules.domain.catalog import RULE_BY_ID, RULES
from core.rules.domain.exceptions import InvalidRuleOrderError

DEFAULT_CUSTOM_ORDER: tuple[str, ...] = ("R2", "R4", "R6", "R1", "R3", "R5")


@dataclass(frozen=True, slots=True)
class CustomOrderStrategy:
    sequence: tuple[str, ...] = DEFAULT_CUSTOM_ORDER
    name: str = field(default="custom")

    def __post_init__(self) -> None:
        known = set(RULE_BY_ID)
        given = set(self.sequence)
        if len(self.sequence) != len(given):
            raise InvalidRuleOrderError("duplicated rule identifiers")
        unknown = given - known
        if unknown:
            raise InvalidRuleOrderError(f"unknown rule identifiers: {sorted(unknown)}")
        missing = known - given
        if missing:
            raise InvalidRuleOrderError(f"missing rule identifiers: {sorted(missing)}")

    def order(self, rules: tuple[TransitionRule, ...]) -> tuple[TransitionRule, ...]:
        position = {rule_id: index for index, rule_id in enumerate(self.sequence)}
        return tuple(sorted(rules, key=lambda rule: position[rule.id]))


def parse_order(raw: str) -> tuple[str, ...]:
    identifiers = tuple(
        token.strip().upper() for token in raw.split(",") if token.strip()
    )
    if len(identifiers) != len(RULES):
        raise InvalidRuleOrderError(
            f"expected {len(RULES)} identifiers, got {len(identifiers)}"
        )
    return CustomOrderStrategy(sequence=identifiers).sequence


CUSTOM = CustomOrderStrategy()

In [ ]:
%%writefile src/core/rules/strategies/descending.py
from __future__ import annotations

from dataclasses import dataclass

from core.rules.domain.base import TransitionRule
from core.rules.domain.catalog import RULES

_CANONICAL_POSITION = {rule.id: position for position, rule in enumerate(RULES)}


@dataclass(frozen=True, slots=True)
class DescendingStrategy:
    name: str = "descending"

    def order(self, rules: tuple[TransitionRule, ...]) -> tuple[TransitionRule, ...]:
        return tuple(
            sorted(rules, key=lambda rule: _CANONICAL_POSITION[rule.id], reverse=True)
        )


DESCENDING = DescendingStrategy()

## core/domain/

In [ ]:
%%writefile src/core/domain/disk.py
from core.rules.domain.base import TOTAL_DISKS, Disk

__all__ = ["TOTAL_DISKS", "Disk"]

In [ ]:
%%writefile src/core/domain/peg.py
from core.rules.domain.base import CAPACITIES, Peg

__all__ = ["CAPACITIES", "Peg"]

In [ ]:
%%writefile src/core/domain/problem.py
"""Estado inicial e catálogo de cartas.

Inicial, fixo pelo aparato:  H1[VERDE, VERMELHO]  H2[AZUL]  H3[]
Cada carta define um objetivo a alcançar a partir dele.

`Problem` valida inicial e objetivo na construção: carta impossível falha na
importação do módulo, não no meio de uma busca. O campo `initial` tem padrão para
que testes montem cenários sintéticos sem tocar no catálogo.

`all_goal_problems` monta uma carta sintética por estado do espaço (G01 a
G36), todas a partir do mesmo inicial: é a bateria usada para comparar os
algoritmos por média e mediana em vez de por uma carta só.

Único ponto a alterar quando o conjunto de cartas muda: os testes usam
propriedades relativas, nunca o literal do objetivo.
"""

from __future__ import annotations

from collections.abc import Mapping
from dataclasses import dataclass
from types import MappingProxyType

from core.domain.state import State, build_state, validate
from core.domain.state_space import all_states
from core.rules.domain.base import Disk

GREEN = Disk.GREEN
RED = Disk.RED
BLUE = Disk.BLUE


class UnknownProblemError(Exception):
    def __init__(self, problem_id: str) -> None:
        super().__init__(f"unknown problem identifier: {problem_id!r}")
        self.problem_id = problem_id


INITIAL_STATE: State = build_state((GREEN, RED), (BLUE,), ())


@dataclass(frozen=True, slots=True)
class Problem:
    id: str
    goal: State
    initial: State = INITIAL_STATE

    def __post_init__(self) -> None:
        validate(self.goal)
        validate(self.initial)

    def is_goal(self, state: State) -> bool:
        return state == self.goal


PROBLEMS: tuple[Problem, ...] = (Problem("P1", build_state((), (RED, GREEN), (BLUE,))),)

PROBLEM_BY_ID: Mapping[str, Problem] = MappingProxyType(
    {problem.id: problem for problem in PROBLEMS}
)

PROBLEM_IDS: tuple[str, ...] = tuple(PROBLEM_BY_ID)


def get_problem(problem_id: str) -> Problem:
    try:
        return PROBLEM_BY_ID[problem_id]
    except KeyError:
        raise UnknownProblemError(problem_id) from None


def all_goal_problems(initial: State = INITIAL_STATE) -> tuple[Problem, ...]:
    return tuple(
        Problem(f"G{index:02d}", goal, initial=initial)
        for index, goal in enumerate(all_states(), start=1)
    )

In [ ]:
%%writefile src/core/domain/state.py
from __future__ import annotations

from collections.abc import Iterable

from core.rules.domain.base import CAPACITIES, TOTAL_DISKS, Disk, Peg, Stack, State

__all__ = [
    "CAPACITIES",
    "EMPTY_STATE",
    "TOTAL_DISKS",
    "Disk",
    "InvalidStateError",
    "Peg",
    "Stack",
    "State",
    "build_state",
    "disks_of",
    "is_valid",
    "stack_of",
    "top_of",
    "validate",
]

EMPTY_STATE: State = ((), (), ())


class InvalidStateError(Exception):
    def __init__(self, reason: str) -> None:
        super().__init__(f"invalid state: {reason}")
        self.reason = reason


def build_state(h1: Iterable[Disk], h2: Iterable[Disk], h3: Iterable[Disk]) -> State:
    state: State = (tuple(h1), tuple(h2), tuple(h3))
    validate(state)
    return state


def validate(state: State) -> None:
    if len(state) != len(Peg):
        raise InvalidStateError(f"expected {len(Peg)} pegs, got {len(state)}")

    for peg in Peg:
        capacity = CAPACITIES[peg]
        if len(state[peg]) > capacity:
            raise InvalidStateError(
                f"peg {peg.name} holds {len(state[peg])} disks, capacity is {capacity}"
            )

    disks = disks_of(state)
    if len(disks) != TOTAL_DISKS:
        raise InvalidStateError(
            f"expected {TOTAL_DISKS} distinct disks, got {len(disks)}"
        )


def is_valid(state: State) -> bool:
    try:
        validate(state)
    except InvalidStateError:
        return False
    return True


def disks_of(state: State) -> frozenset[Disk]:
    return frozenset(disk for stack in state for disk in stack)


def stack_of(state: State, peg: Peg) -> Stack:
    return state[peg]


def top_of(state: State, peg: Peg) -> Disk | None:
    stack = state[peg]
    return stack[-1] if stack else None

In [ ]:
%%writefile src/core/domain/state_space.py
"""Enumeração exaustiva do espaço de estados.

Não participa de busca alguma: descreve o espaço por força bruta e serve de
oráculo independente aos testes.

Contagem:
    3! = 6 permutações x 10 cortes em três partes = 60 disposições
    as capacidades (3, 2, 1) deixam 6 distribuições:
        (3,0,0)  (2,1,0)  (2,0,1)  (1,2,0)  (1,1,1)  (0,2,1)
    6 distribuições x 6 arranjos = 36 estados válidos

Propriedades:
    - Grafo não dirigido: toda regra tem inversa exata.
    - Grafo conexo: todo objetivo é alcançável de qualquer inicial. Logo
      backtracking e largura sempre terminam em SUCESSO neste problema; só a
      busca irrevogável pode terminar em IMPASSE.
    - Fator de ramificação: 2 a 4 regras aplicáveis por estado.
    - Profundidade máxima de um caminho sem repetição: 36.

`shortest_distance` é uma busca em largura escrita à parte do motor, sem árvore,
nó ou estratégia. É o oráculo contra o qual a otimalidade de `breadth_first` é
verificada. `cheapest_cost` faz o mesmo papel para a busca ordenada: um
Dijkstra sobre o custo das regras, também fora do motor.
"""

from __future__ import annotations

import heapq
from collections import deque
from collections.abc import Iterator
from itertools import permutations

from core.domain.state import Disk, State, is_valid
from core.rules.domain.base import TransitionRule
from core.rules.domain.catalog import RULES


def all_states() -> tuple[State, ...]:
    return tuple(_generate_states())


def _generate_states() -> Iterator[State]:
    disks = tuple(Disk)
    for arrangement in permutations(disks):
        for first in range(len(disks) + 1):
            for second in range(first, len(disks) + 1):
                candidate: State = (
                    arrangement[:first],
                    arrangement[first:second],
                    arrangement[second:],
                )
                if is_valid(candidate):
                    yield candidate


def applicable_rules(state: State) -> tuple[TransitionRule, ...]:
    return tuple(rule for rule in RULES if rule.is_applicable(state))


def successors(state: State) -> tuple[tuple[TransitionRule, State], ...]:
    return tuple((rule, rule.apply(state)) for rule in applicable_rules(state))


def reachable_from(source: State) -> frozenset[State]:
    seen = {source}
    queue = deque([source])
    while queue:
        current = queue.popleft()
        for _, successor in successors(current):
            if successor not in seen:
                seen.add(successor)
                queue.append(successor)
    return frozenset(seen)


def shortest_distance(source: State, target: State) -> int | None:
    if source == target:
        return 0
    depths = {source: 0}
    queue = deque([source])
    while queue:
        current = queue.popleft()
        for _, successor in successors(current):
            if successor in depths:
                continue
            depths[successor] = depths[current] + 1
            if successor == target:
                return depths[successor]
            queue.append(successor)
    return None


def cheapest_cost(source: State, target: State) -> int | None:
    costs = {source: 0}
    heap: list[tuple[int, int, State]] = [(0, 0, source)]
    pushed = 1
    while heap:
        cost, _, current = heapq.heappop(heap)
        if current == target:
            return cost
        if cost > costs[current]:
            continue
        for rule, successor in successors(current):
            candidate = cost + rule.cost
            if candidate < costs.get(successor, candidate + 1):
                costs[successor] = candidate
                heapq.heappush(heap, (candidate, pushed, successor))
                pushed += 1
    return None

## core/search_tree/

In [ ]:
%%writefile src/core/search_tree/frontier.py
"""FRONTEIRA — a lista de ABERTOS.

Os nós já gerados que ainda não foram processados. A busca tira um daqui,
olha em volta e devolve os filhos. Fronteira vazia, busca encerrada.

Quem sai primeiro define o algoritmo: pilha afunda, fila varre por nível,
fila de prioridade sai pelo menor custo.
"""

from __future__ import annotations

import heapq
from collections import deque
from typing import Protocol

from core.search_tree.node import Node


class EmptyFrontierError(Exception):
    """Tentou tirar um nó de uma fronteira vazia."""

    def __init__(self) -> None:
        super().__init__("cannot pop from an empty frontier")


class Frontier(Protocol):
    """Guardar um nó, tirar um nó, contar quantos faltam. Só isso.

    Sem índice, sem busca por estado, sem percorrer o conteúdo: o único
    acesso é pela ponta. O contrato não diz QUAL nó o pop devolve — é a
    única coisa que muda entre as implementações, e é o que define a busca.
    """

    def push(self, node: Node) -> None:
        """Guarda um nó. Ele sempre entra pelo fim."""
        ...

    def pop(self) -> Node:
        """Tira um nó e devolve. QUAL nó é o que muda entre as fronteiras."""
        ...

    def __len__(self) -> int:
        """Quantos nós ainda esperam. Zero encerra a busca."""
        ...


class StackFrontier:
    """PILHA (LIFO): sai o último que entrou. Usada pelo backtracking.

    ```
    push S1, S2, S3  ->  [S1 S2 S3]  ->  pop devolve S3
    ```

    Entra e sai pelo mesmo lado, então o filho recém-gerado é sempre o
    próximo a ser olhado: a busca afunda num ramo até o fim.
    """

    __slots__ = ("_nodes",)

    def __init__(self) -> None:
        self._nodes: list[Node] = []

    def push(self, node: Node) -> None:
        """Empilha o nó no topo."""
        self._nodes.append(node)

    def pop(self) -> Node:
        """Tira e devolve o nó do topo: o ÚLTIMO que entrou."""
        if not self._nodes:
            raise EmptyFrontierError
        return self._nodes.pop()

    def __len__(self) -> int:
        """Quantos nós estão empilhados."""
        return len(self._nodes)


class QueueFrontier:
    """FILA (FIFO): sai o primeiro que entrou. Usada pela busca em largura.

    ```
    push S1, S2, S3  ->  [S1 S2 S3]  ->  pop devolve S1
    ```

    Entra por um lado, sai pelo outro, então os filhos esperam atrás de todo
    o nível atual: a busca varre nível por nível e acha o caminho mais curto.

    Usa deque, não list, porque popleft é O(1) e list.pop(0) é O(n).
    """

    __slots__ = ("_nodes",)

    def __init__(self) -> None:
        self._nodes: deque[Node] = deque()

    def push(self, node: Node) -> None:
        """Enfileira o nó no fim da fila."""
        self._nodes.append(node)

    def pop(self) -> Node:
        """Tira e devolve o nó do início: o PRIMEIRO que entrou.

        É a única linha que difere da StackFrontier, e é ela que troca
        profundidade por largura.
        """
        if not self._nodes:
            raise EmptyFrontierError
        return self._nodes.popleft()

    def __len__(self) -> int:
        """Quantos nós estão na fila."""
        return len(self._nodes)


class PriorityFrontier:
    """FILA ORDENADA POR CUSTO: sai o de menor custo acumulado. Usada pela
    busca ordenada.

    ```
    push S1(5), S2(6), S3(6), S4(1)  ->  pop devolve S4, S1, S2, S3
    ```

    Empate de custo sai o gerado primeiro (menor `order`). Irmãos de mesmo
    custo nascem na ordem da estratégia, então ela é o desempate entre eles.

    Heap de (custo, ordem, nó): a ordem é única, então o nó nunca é comparado.
    """

    __slots__ = ("_heap", "_removed")

    def __init__(self) -> None:
        self._heap: list[tuple[int, int, Node]] = []
        self._removed: set[int] = set()

    def push(self, node: Node) -> None:
        """Insere o nó na posição do seu custo."""
        heapq.heappush(self._heap, (node.cost, node.order, node))

    def pop(self) -> Node:
        """Tira e devolve o nó de MENOR custo; no empate, o mais antigo."""
        while self._heap:
            _, order, node = heapq.heappop(self._heap)
            if order in self._removed:
                self._removed.discard(order)
                continue
            return node
        raise EmptyFrontierError

    def remove(self, node: Node) -> None:
        """Tira um nó que ainda está na fila, sem devolvê-lo.

        A remoção é preguiçosa: o nó só é descartado quando chegaria a vez
        dele no pop. O nó precisa estar na fila.
        """
        self._removed.add(node.order)

    def __len__(self) -> int:
        """Quantos nós ainda esperam, sem contar os removidos."""
        return len(self._heap) - len(self._removed)

In [ ]:
%%writefile src/core/search_tree/metrics.py
from __future__ import annotations

from dataclasses import dataclass, field


@dataclass(frozen=True, slots=True)
class SearchMetrics:
    iterations: int
    nodes_generated: int
    nodes_visited: int
    rules_tested: int
    backtracks: int
    deadlocks: int
    max_depth: int
    elapsed_ms: float


@dataclass(slots=True)
class MetricsCollector:
    iterations: int = 0
    nodes_generated: int = 0
    nodes_visited: int = 0
    rules_tested: int = 0
    backtracks: int = 0
    deadlocks: int = 0
    max_depth: int = field(default=0)

    def count_iteration(self) -> None:
        self.iterations += 1

    def count_generated(self, depth: int) -> None:
        self.nodes_generated += 1
        self.max_depth = max(self.max_depth, depth)

    def count_visited(self) -> None:
        self.nodes_visited += 1

    def count_rule_test(self) -> None:
        self.rules_tested += 1

    def count_backtrack(self) -> None:
        self.backtracks += 1

    def count_deadlock(self) -> None:
        self.deadlocks += 1

    def seal(self, elapsed_ms: float) -> SearchMetrics:
        return SearchMetrics(
            iterations=self.iterations,
            nodes_generated=self.nodes_generated,
            nodes_visited=self.nodes_visited,
            rules_tested=self.rules_tested,
            backtracks=self.backtracks,
            deadlocks=self.deadlocks,
            max_depth=self.max_depth,
            elapsed_ms=elapsed_ms,
        )

In [ ]:
%%writefile src/core/search_tree/node.py
from __future__ import annotations

from dataclasses import dataclass

from core.domain.state import State
from core.rules.domain.base import TransitionRule


@dataclass(frozen=True, slots=True)
class Node:
    state: State
    parent: Node | None
    rule: TransitionRule | None
    depth: int
    order: int
    cost: int = 0

    @property
    def is_root(self) -> bool:
        return self.parent is None

    def __hash__(self) -> int:
        return hash((self.order, self.state))

In [ ]:
%%writefile src/core/search_tree/outcome.py
from __future__ import annotations

from enum import Enum, unique


@unique
class Outcome(Enum):
    SUCCESS = "SUCCESS"
    DEADLOCK = "DEADLOCK"
    CUTOFF = "CUTOFF"
    FAILURE = "FAILURE"

    @property
    def is_success(self) -> bool:
        return self is Outcome.SUCCESS

In [ ]:
%%writefile src/core/search_tree/path.py
from __future__ import annotations

from collections.abc import Iterator

from core.domain.state import State
from core.search_tree.node import Node


def ancestors(node: Node) -> Iterator[Node]:
    current = node.parent
    while current is not None:
        yield current
        current = current.parent


def path_to(node: Node) -> tuple[Node, ...]:
    reversed_path = [node, *ancestors(node)]
    reversed_path.reverse()
    return tuple(reversed_path)


def states_on_path(node: Node) -> frozenset[State]:
    return frozenset(step.state for step in path_to(node))


def contains_state(node: Node, state: State) -> bool:
    if node.state == state:
        return True
    return any(ancestor.state == state for ancestor in ancestors(node))


def applied_rules(node: Node) -> tuple[str, ...]:
    return tuple(step.rule.id for step in path_to(node) if step.rule is not None)

In [ ]:
%%writefile src/core/search_tree/result.py
from __future__ import annotations

from dataclasses import dataclass

from core.search_tree.metrics import SearchMetrics
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome
from core.search_tree.trace import Trace


@dataclass(frozen=True, slots=True)
class SearchResult:
    algorithm: str
    strategy: str
    problem: str
    outcome: Outcome
    solution_path: tuple[Node, ...]
    applied_rules: tuple[str, ...]
    metrics: SearchMetrics
    trace: Trace

    @property
    def solution_length(self) -> int | None:
        return len(self.applied_rules) if self.outcome.is_success else None

    @property
    def solution_cost(self) -> int | None:
        return self.solution_path[-1].cost if self.outcome.is_success else None

    @property
    def label(self) -> str:
        return f"{self.algorithm} · {self.strategy}"

In [ ]:
%%writefile src/core/search_tree/trace.py
from __future__ import annotations

from collections.abc import Iterator
from dataclasses import dataclass, field
from enum import Enum, unique

from core.domain.state import State
from core.search_tree.node import Node


@unique
class TraceEvent(Enum):
    ROOT = "ROOT"
    VISIT = "VISIT"
    GENERATE = "GENERATE"
    PRUNE = "PRUNE"
    GOAL = "GOAL"
    DEADLOCK = "DEADLOCK"
    BACKTRACK = "BACKTRACK"
    CUTOFF = "CUTOFF"
    EXHAUSTED = "EXHAUSTED"


@dataclass(frozen=True, slots=True)
class TraceStep:
    iteration: int
    event: TraceEvent
    node_order: int
    parent_order: int | None
    depth: int
    rule_id: str | None
    state: State


@dataclass(frozen=True, slots=True)
class Trace:
    steps: tuple[TraceStep, ...] = ()

    def __iter__(self) -> Iterator[TraceStep]:
        return iter(self.steps)

    def __len__(self) -> int:
        return len(self.steps)

    def of_event(self, event: TraceEvent) -> tuple[TraceStep, ...]:
        return tuple(step for step in self.steps if step.event is event)


@dataclass(slots=True)
class TraceRecorder:
    _steps: list[TraceStep] = field(default_factory=list)

    def record(
        self,
        *,
        iteration: int,
        event: TraceEvent,
        node: Node,
        rule_id: str | None = None,
    ) -> None:
        parent = node.parent
        self._steps.append(
            TraceStep(
                iteration=iteration,
                event=event,
                node_order=node.order,
                parent_order=None if parent is None else parent.order,
                depth=node.depth,
                rule_id=rule_id if rule_id is not None else _rule_id_of(node),
                state=node.state,
            )
        )

    def record_prune(
        self, *, iteration: int, node: Node, rule_id: str, state: State
    ) -> None:
        self._steps.append(
            TraceStep(
                iteration=iteration,
                event=TraceEvent.PRUNE,
                node_order=node.order,
                parent_order=None if node.parent is None else node.parent.order,
                depth=node.depth,
                rule_id=rule_id,
                state=state,
            )
        )

    def seal(self) -> Trace:
        return Trace(steps=tuple(self._steps))


def _rule_id_of(node: Node) -> str | None:
    return None if node.rule is None else node.rule.id

In [ ]:
%%writefile src/core/search_tree/tree.py
from __future__ import annotations

from collections.abc import Iterator

from core.domain.state import State
from core.rules.domain.base import TransitionRule
from core.rules.domain.catalog import RULES
from core.search_tree.node import Node
from core.search_tree.path import ancestors


class SearchTree:
    __slots__ = ("_nodes", "_rules")

    def __init__(self, rules: tuple[TransitionRule, ...] = RULES) -> None:
        self._rules = rules
        self._nodes: list[Node] = []

    @property
    def rules(self) -> tuple[TransitionRule, ...]:
        return self._rules

    @property
    def nodes(self) -> tuple[Node, ...]:
        return tuple(self._nodes)

    def root(self, state: State) -> Node:
        self._nodes = []
        node = Node(state=state, parent=None, rule=None, depth=0, order=0)
        self._nodes.append(node)
        return node

    def expand(self, node: Node, rule: TransitionRule) -> Node:
        child = Node(
            state=rule.apply(node.state),
            parent=node,
            rule=rule,
            depth=node.depth + 1,
            order=len(self._nodes),
            cost=node.cost + rule.cost,
        )
        self._nodes.append(child)
        return child

    def ancestors(self, node: Node) -> Iterator[Node]:
        return ancestors(node)

## core/algorithms/domain/

In [ ]:
%%writefile src/core/algorithms/domain/base.py
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from time import perf_counter
from typing import ClassVar

from config import settings
from core.domain.problem import Problem
from core.domain.state import State
from core.rules.domain.base import TransitionRule
from core.rules.strategies.domain.base import ControlStrategy
from core.search_tree import path
from core.search_tree.metrics import MetricsCollector
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome
from core.search_tree.result import SearchResult
from core.search_tree.trace import TraceEvent, TraceRecorder
from core.search_tree.tree import SearchTree


class SearchNotStartedError(Exception):
    def __init__(self) -> None:
        super().__init__("search context is only available during solve()")


@dataclass(slots=True)
class _RunContext:
    problem: Problem
    root: Node
    metrics: MetricsCollector = field(default_factory=MetricsCollector)
    trace: TraceRecorder = field(default_factory=TraceRecorder)
    goal_node: Node | None = None


class SearchAlgorithm(ABC):
    """Motor compartilhado pelos algoritmos de busca.

    Visitar, gerar, podar, contar e registrar vivem aqui. Cada algoritmo
    implementa só o próprio laço, em `_search`.
    """

    name: ClassVar[str] = "abstract"

    def __init__(
        self,
        tree: SearchTree,
        strategy: ControlStrategy,
        *,
        max_iterations: int = settings.MAX_ITERATIONS,
    ) -> None:
        self._tree = tree
        self._strategy = strategy
        self._max_iterations = max_iterations
        self._context: _RunContext | None = None

    @property
    def strategy(self) -> ControlStrategy:
        return self._strategy

    @property
    def max_iterations(self) -> int:
        return self._max_iterations

    def solve(self, problem: Problem) -> SearchResult:
        """Porta de entrada da busca.

        Monta o contexto da execução, cronometra o `_search` do algoritmo
        concreto e sela métricas, trace e caminho no resultado.
        """
        context = _RunContext(problem=problem, root=self._tree.root(problem.initial))
        self._context = context
        context.metrics.count_generated(context.root.depth)
        context.trace.record(iteration=0, event=TraceEvent.ROOT, node=context.root)

        started = perf_counter()
        try:
            outcome = self._search(problem)
        finally:
            elapsed_ms = (perf_counter() - started) * 1000.0
            self._context = None

        goal_node = context.goal_node if outcome.is_success else None
        solution = () if goal_node is None else path.path_to(goal_node)
        return SearchResult(
            algorithm=self.name,
            strategy=self._strategy.name,
            problem=problem.id,
            outcome=outcome,
            solution_path=solution,
            applied_rules=() if goal_node is None else path.applied_rules(goal_node),
            metrics=context.metrics.seal(elapsed_ms),
            trace=context.trace.seal(),
        )

    @abstractmethod
    def _search(self, problem: Problem) -> Outcome:
        """O laço de cada algoritmo. A única coisa que eles não compartilham."""
        ...

    def _applicable_rules(self, node: Node) -> tuple[TransitionRule, ...]:
        """As regras que este nó pode usar, prontas para serem tentadas.

        Faz os três passos em ordem: testa quais são aplicáveis, ordena pela
        estratégia e poda as que repetiriam um estado.
        """
        context = self._require_context()
        applicable: list[TransitionRule] = []
        for rule in self._tree.rules:
            context.metrics.count_rule_test()
            if rule.is_applicable(node.state):
                applicable.append(rule)

        allowed: list[TransitionRule] = []
        for rule in self._strategy.order(tuple(applicable)):
            successor = rule.apply(node.state)
            if self._is_repetition(node, rule, successor):
                context.trace.record_prune(
                    iteration=context.metrics.iterations,
                    node=node,
                    rule_id=rule.id,
                    state=successor,
                )
                continue
            allowed.append(rule)
        return tuple(allowed)

    def _is_repetition(
        self, node: Node, rule: TransitionRule, successor: State
    ) -> bool:
        """A política de repetição: aqui, um estado já presente no caminho.

        A busca em largura sobrescreve para usar o conjunto global de
        fechados, que é uma poda mais forte. A ordenada sobrescreve para
        comparar o custo do novo caminho com o menor já conhecido.
        """
        return path.contains_state(node, successor)

    def _require_context(self) -> _RunContext:
        """O contexto da execução. Só existe durante o `solve`."""
        if self._context is None:
            raise SearchNotStartedError
        return self._context

    def _next_iteration(self) -> bool:
        """Conta mais uma iteração e aplica o teto.

        Devolve `False` quando o limite estourou, e aí a busca encerra em
        LIMITE. Chame no topo do laço, antes de qualquer trabalho.
        """
        context = self._require_context()
        if context.metrics.iterations >= self._max_iterations:
            context.trace.record(
                iteration=context.metrics.iterations,
                event=TraceEvent.CUTOFF,
                node=context.root,
            )
            return False
        context.metrics.count_iteration()
        return True

    def _visit(self, node: Node) -> None:
        """Só contabilidade: registra que o nó foi olhado. Uma vez por nó."""
        context = self._require_context()
        context.metrics.count_visited()
        context.trace.record(
            iteration=context.metrics.iterations, event=TraceEvent.VISIT, node=node
        )

    def _expand(self, node: Node, rule: TransitionRule) -> Node:
        """Aplica a regra e cria o nó filho.

        É o único ponto do motor que faz a árvore crescer.
        """
        context = self._require_context()
        child = self._tree.expand(node, rule)
        context.metrics.count_generated(child.depth)
        context.trace.record(
            iteration=context.metrics.iterations, event=TraceEvent.GENERATE, node=child
        )
        return child

    def _succeed(self, node: Node) -> Outcome:
        """Guarda o nó objetivo, de onde o caminho solução é reconstruído."""
        context = self._require_context()
        context.goal_node = node
        context.trace.record(
            iteration=context.metrics.iterations, event=TraceEvent.GOAL, node=node
        )
        return Outcome.SUCCESS

    def _deadlock(self, node: Node) -> None:
        """Só contabilidade: este nó nasceu sem saída.

        Não encerra busca alguma. Quem decide o que fazer com o impasse é o
        algoritmo: a irrevogável desiste, o backtracking retrocede.
        """
        context = self._require_context()
        context.metrics.count_deadlock()
        context.trace.record(
            iteration=context.metrics.iterations, event=TraceEvent.DEADLOCK, node=node
        )

    def _backtrack(self, node: Node) -> None:
        """Só contabilidade: um retrocesso aconteceu.

        Nada é desfeito aqui. O retrocesso em si é o algoritmo deixando de
        devolver o nó à pilha.
        """
        context = self._require_context()
        context.metrics.count_backtrack()
        context.trace.record(
            iteration=context.metrics.iterations, event=TraceEvent.BACKTRACK, node=node
        )

    def _exhausted(self) -> Outcome:
        """Fim de linha: nada mais a explorar e nenhum objetivo encontrado."""
        context = self._require_context()
        context.trace.record(
            iteration=context.metrics.iterations,
            event=TraceEvent.EXHAUSTED,
            node=context.root,
        )
        return Outcome.FAILURE

In [ ]:
%%writefile src/core/algorithms/domain/registry.py
from __future__ import annotations

from collections.abc import Mapping
from types import MappingProxyType

from core.algorithms.backtracking import BacktrackingSearch
from core.algorithms.breadth_first import BreadthFirstSearch
from core.algorithms.domain.base import SearchAlgorithm
from core.algorithms.irrevocable import IrrevocableSearch
from core.algorithms.ordered import OrderedSearch


class UnknownAlgorithmError(Exception):
    def __init__(self, name: str) -> None:
        super().__init__(f"unknown search algorithm: {name!r}")
        self.name = name


ALGORITHMS: Mapping[str, type[SearchAlgorithm]] = MappingProxyType(
    {
        IrrevocableSearch.name: IrrevocableSearch,
        BacktrackingSearch.name: BacktrackingSearch,
        BreadthFirstSearch.name: BreadthFirstSearch,
        OrderedSearch.name: OrderedSearch,
    }
)

ALGORITHM_NAMES: tuple[str, ...] = tuple(ALGORITHMS)


def get_algorithm(name: str) -> type[SearchAlgorithm]:
    try:
        return ALGORITHMS[name]
    except KeyError:
        raise UnknownAlgorithmError(name) from None

## core/algorithms/

In [ ]:
%%writefile src/core/algorithms/backtracking.py
"""BUSCA COM BACKTRACKING

1. Estado inicial e objetivo (carta P1)

    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])

2. Critério

    A mesma descida da busca irrevogável, mas as regras válidas de cada
    estado ficam guardadas numa pilha. Quando um estado não tem mais regra,
    ele sai da pilha (retrocesso) e o estado anterior tenta a próxima regra
    que tinha guardado. Uma regra que leva a um estado já no caminho atual
    é descartada (poda).

3. Execução com a ordem decrescente R6 → R5 → R4 → R3 → R2 → R1

    Passo 1   S0 = ([V, R], [A], [])
              guarda [R4, R3, R2, R1], aplica R4
              ↓
    Passo 2   S1 = ([V, R], [], [A])
              R6 volta a S0: poda.  Guarda [R5, R1], aplica R5
              ↓
    Passo 3   S2 = ([V, R, A], [], [])
              R2 volta a S1 e R1 volta a S0: poda.  Nada guardado.
              ✗ impasse → retrocesso: S2 sai da pilha
              ↑
    Passo 4   S1 = ([V, R], [], [A])   ainda tem [R1], aplica R1
              ↓
    Passo 5   S3 = ([V], [R], [A])
              R3 volta a S1: poda.  Guarda [R6, R5, R1], aplica R6
              ↓
              ...  continua descendo; a cada beco sem saída, um retrocesso

    Passo 51  S36 = ([], [R, V], [A])
              ✓ SUCESSO

4. Árvore de busca (início)

    S0 ([V,R], [A], [])
    └── R4 → S1 ([V,R], [], [A])
        ├── R5 → S2 ([V,R,A], [], [])   (impasse, retrocesso)
        └── R1 → S3 ([V], [R], [A])
            └── R6 → S4 ([V], [R,A], [])
                └── R3 → S5 ([V,A], [R], [])
                    └── R4 → S6 ([V,A], [], [R])
                        ├── R5 → S7 ([V,A,R], [], [])   (impasse, retrocesso)
                        └── R1 → S8 ([V], [A], [R])
                            └── ...  até S36 = Sf

5. Caminho encontrado

    R4 R1 R6 R3 R4 R1 R6 R2 R3 R6 R3 R4 R1 R6 R2 R3 R6 R3 R4 R1 R6 R2

    Número de movimentos: 22       Iterações: 51
    Retrocessos: 14                Impasses: 8

    Com a ordem crescente: 14 movimentos em 15 iterações, sem retrocesso.
    O caminho ótimo tem 3.

6. Conclusão

    O impasse da busca irrevogável vira um desvio: nenhum ramo é abandonado
    sem ser explorado e, como todo estado alcança todo estado, a solução
    sempre aparece. Só não é a melhor. A busca para no primeiro objetivo
    que encontra, tenha o caminho o comprimento que tiver.
"""

from __future__ import annotations

from collections import deque
from typing import ClassVar

from core.algorithms.domain.base import SearchAlgorithm
from core.domain.problem import Problem
from core.rules.domain.base import TransitionRule
from core.search_tree.frontier import StackFrontier
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome


class BacktrackingSearch(SearchAlgorithm):
    name: ClassVar[str] = "backtracking"

    def _search(self, problem: Problem) -> Outcome:
        """Desce enquanto houver regra; sem regra, retrocede.

        Cada iteração olha a ponta do caminho e faz uma coisa OU a outra:
        desce, e o nó continua no caminho com um filho por cima; ou retrocede,
        e o nó sai do caminho, deixando o pai como ponta.

        Sempre encontra solução, se existir — mas é a primeira que aparecer,
        não a mais curta.
        """
        context = self._require_context()

        path = StackFrontier()
        path.push(context.root)

        # Para cada nó do caminho, as regras que ele ainda não tentou. Sem
        # isso não há retrocesso: o nó não saberia por onde continuar.
        untried: dict[int, deque[TransitionRule]] = {}

        while len(path):
            if not self._next_iteration():
                return Outcome.CUTOFF

            # Topo da pilha = ponta do caminho. Ele só volta para a pilha se
            # ainda tiver regra -- ver o if lá embaixo.
            node = path.pop()

            if problem.is_goal(node.state):
                return self._succeed(node)

            remaining = self._remaining_rules(node, untried)
            if remaining:
                # DESCE: o nó continua no caminho e o filho entra por cima.
                path.push(node)
                path.push(self._expand(node, remaining.popleft()))
            else:
                # RETROCEDE: sem regras, o nó não volta e sai do caminho. Na
                # próxima iteração o topo é o pai, que tenta a regra seguinte.
                self._backtrack(node)

        return self._exhausted()

    def _remaining_rules(
        self, node: Node, untried: dict[int, deque[TransitionRule]]
    ) -> deque[TransitionRule]:
        """Na primeira visita monta a lista do nó; nas voltas seguintes apenas
        devolve o que sobrou dela. _applicable_rules já poda a regra que
        repetiria um estado do caminho.

        As seis regras que entram nessa lista:

        ```
        regra | de | para | inversa
        ------+----+------+--------
        R1    | H1 | H2   | R3
        R2    | H1 | H3   | R5
        R3    | H2 | H1   | R1
        R4    | H2 | H3   | R6
        R5    | H3 | H1   | R2
        R6    | H3 | H2   | R4
        ```

        Toda regra tem inversa exata, e é daí que vem a necessidade da poda:
        sem ela, R1 seguido de R3 devolveria a busca ao mesmo estado, sempre.
        """
        if node.order not in untried:
            self._visit(node)
            candidates = self._applicable_rules(node)
            untried[node.order] = deque(candidates)
            if not candidates:
                self._deadlock(node)
        return untried[node.order]

In [ ]:
%%writefile src/core/algorithms/breadth_first.py
"""BUSCA EM LARGURA

1. Estado inicial e objetivo (carta P1)

    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])

2. Critério

    As regras são analisadas na ordem crescente R1 → R2 → R3 → R4 → R5 → R6.
    Em vez de escolher uma, a busca aplica todas as válidas e coloca os
    filhos no fim de ABERTOS. O estado visitado vai para FECHADOS. Uma
    regra cujo resultado já está em ABERTOS ou FECHADOS é descartada (poda).

3. Execução

    Nível 0
        ABERTOS = [S0]               FECHADOS = []

    Expansão de S0 = ([V, R], [A], [])
        R1 → S1 = ([V], [A, R], [])
        R2 → S2 = ([V], [A], [R])
        R3 → S3 = ([V, R, A], [], [])
        R4 → S4 = ([V, R], [], [A])
        R5, R6: inválidas, H3 vazia
        ABERTOS = [S1, S2, S3, S4]   FECHADOS = [S0]

    Nível 1

    Expansão de S1 = ([V], [A, R], [])
        R1: inválida, H2 cheia
        R2 → S5 = ([], [A, R], [V])
        R3 → S0: poda
        R4 → S2: poda
        ABERTOS = [S2, S3, S4, S5]   FECHADOS = [S0, S1]

    Expansão de S2 = ([V], [A], [R])
        R1 → S6 = ([], [A, V], [R])
        R3 → S7 = ([V, A], [], [R])
        R5 → S0, R6 → S1: poda
        ABERTOS = [S3, S4, S5, S6, S7]

    Expansão de S3 = ([V, R, A], [], [])
        R1 → S0, R2 → S4: poda.  Nada novo.

    Expansão de S4 = ([V, R], [], [A])
        R1 → S8 = ([V], [R], [A])
        R5 → S3, R6 → S0: poda
        ABERTOS = [S5, S6, S7, S8]   FECHADOS = [S0, S1, S2, S3, S4]

    Nível 2
        S5 gera S9.  S6 gera S10.  S7 gera S11 e S12.
        S8 = ([V], [R], [A]) gera:
            R1 → S13 = ([], [R, V], [A])   ← é Sf, entra na fila
            R6 → S14 = ([V], [R, A], [])
        ABERTOS = [S9, S10, S11, S12, S13, S14]

    Nível 3
        S9 gera S15 e S16.  S10, S11 e S12 só geram repetidos.
        Chega a vez de S13 = ([], [R, V], [A])
        ✓ SUCESSO

4. Árvore de busca

    S0 ([V,R], [A], [])
    ├── R1 → S1 ([V], [A,R], [])
    │   └── R2 → S5 ([], [A,R], [V])
    │       └── R3 → S9 ([R], [A], [V])
    │           ├── R3 → S15
    │           └── R5 → S16
    ├── R2 → S2 ([V], [A], [R])
    │   ├── R1 → S6 ([], [A,V], [R])
    │   │   └── R5 → S10
    │   └── R3 → S7 ([V,A], [], [R])
    │       ├── R5 → S11
    │       └── R6 → S12
    ├── R3 → S3 ([V,R,A], [], [])
    └── R4 → S4 ([V,R], [], [A])
        └── R1 → S8 ([V], [R], [A])
            ├── R1 → S13 ([], [R,V], [A])   (objetivo)
            └── R6 → S14 ([V], [R,A], [])

5. Caminho solução

    S0 ([V,R], [A], [])
     │ R4: azul H2 → H3
     ↓
    S4 ([V,R], [], [A])
     │ R1: vermelho H1 → H2
     ↓
    S8 ([V], [R], [A])
     │ R1: verde H1 → H2
     ↓
    Sf ([], [R,V], [A])

    Número de movimentos: 3      Iterações: 14      Gerados: 17

6. Conclusão

    Nada de profundidade d+1 é visitado antes de esgotar a profundidade d,
    então o primeiro caminho até um estado é o mais curto até ele. O objetivo
    encontrado é o ótimo em movimentos, e por isso este método serve de
    referência de comprimento para os outros. A estratégia só muda a ordem
    dentro de um nível.

    A poda aqui é global: um estado descoberto por qualquer ramo nunca é
    gerado de novo. A irrevogável e o backtracking só evitam repetir dentro
    do próprio caminho.
"""

from __future__ import annotations

from typing import ClassVar

from config import settings
from core.algorithms.domain.base import SearchAlgorithm
from core.domain.problem import Problem
from core.domain.state import State
from core.rules.domain.base import TransitionRule
from core.rules.strategies.domain.base import ControlStrategy
from core.search_tree.frontier import QueueFrontier
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome
from core.search_tree.tree import SearchTree


class BreadthFirstSearch(SearchAlgorithm):
    name: ClassVar[str] = "breadth_first"

    def __init__(
        self,
        tree: SearchTree,
        strategy: ControlStrategy,
        *,
        max_iterations: int = settings.MAX_ITERATIONS,
    ) -> None:
        super().__init__(tree, strategy, max_iterations=max_iterations)
        self._closed: set[State] = set()
        self._open_states: set[State] = set()

    def _search(self, problem: Problem) -> Outcome:
        context = self._require_context()
        frontier = QueueFrontier()
        frontier.push(context.root)

        self._closed = set()
        self._open_states = {context.root.state}

        while len(frontier):
            if not self._next_iteration():
                return Outcome.CUTOFF

            node = frontier.pop()
            self._open_states.discard(node.state)
            if problem.is_goal(node.state):
                return self._succeed(node)

            self._closed.add(node.state)
            self._visit(node)
            for rule in self._applicable_rules(node):
                child = self._expand(node, rule)
                self._open_states.add(child.state)
                frontier.push(child)

        return self._exhausted()

    def _is_repetition(
        self, node: Node, rule: TransitionRule, successor: State
    ) -> bool:
        return successor in self._closed or successor in self._open_states

In [ ]:
%%writefile src/core/algorithms/ordered.py
"""BUSCA ORDENADA

1. Estado inicial e objetivo (carta P1)

    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])

2. Custos

    Cada regra custa a distância entre as hastes que ela liga:

        R1 = H1 → H2: 1     R3 = H2 → H1: 1     R4 = H2 → H3: 1
        R2 = H1 → H3: 2     R5 = H3 → H1: 2     R6 = H3 → H2: 1

    O custo de um nó é a soma das regras do caminho da raiz até ele.

3. Critério

    ABERTOS é ordenada pelo MENOR custo. No empate, sai o nó gerado primeiro;
    entre irmãos de mesmo custo, decide a estratégia (aqui, a ordem crescente
    R1 → R2 → R3 → R4 → R5 → R6). A busca só encerra quando o objetivo vira
    o estado atual, não quando ele é gerado.

    Poda pelo vetor de menor custo: o filho cujo estado já foi gerado com
    custo menor ou igual é descartado. Se o novo custo for menor, o nó
    antigo sai de ABERTOS e da árvore e o novo entra no lugar.

4. Execução

    Expansão de S0 = ([V, R], [A], []), custo 0
        R1 → S1 = ([V], [A, R], [])      1
        R2 → S2 = ([V], [A], [R])        2
        R3 → S3 = ([V, R, A], [], [])    1
        R4 → S4 = ([V, R], [], [A])      1
        R5, R6: inválidas, H3 vazia
        ABERTOS  = [S1(1), S3(1), S4(1), S2(2)]
        FECHADOS = [S0(0)]

    Expansão de S1 = ([V], [A, R], []), custo 1
        R2 → S5 = ([], [A, R], [V])      3
        R3 → S0 (2 ≥ 0): poda
        R4 → estado de S2 (2 ≥ 2): poda
        ABERTOS  = [S3(1), S4(1), S2(2), S5(3)]
        FECHADOS = [S0(0), S1(1)]

    Expansão de S3 = ([V, R, A], [], []), custo 1
        R1 → S0, R2 → S4: poda.  Nada novo.
        ✗ impasse: S3 só vai para FECHADOS
        ABERTOS  = [S4(1), S2(2), S5(3)]
        FECHADOS = [S0(0), S1(1), S3(1)]

    Expansão de S4 = ([V, R], [], [A]), custo 1
        R1 → S6 = ([V], [R], [A])        2
        R5 → S3, R6 → S0: poda
        ABERTOS  = [S2(2), S6(2), S5(3)]
        FECHADOS = [S0(0), S1(1), S3(1), S4(1)]

    Expansão de S2 = ([V], [A], [R]), custo 2
        R1 → S7 = ([], [A, V], [R])      3
        R3 → S8 = ([V, A], [], [R])      3
        R5 → S0, R6 → S1: poda
        ABERTOS  = [S6(2), S5(3), S7(3), S8(3)]
        FECHADOS = [S0(0), S1(1), S3(1), S4(1), S2(2)]

    Expansão de S6 = ([V], [R], [A]), custo 2
        R1 → S9  = ([], [R, V], [A])     3   ← é Sf, mas só entra na fila
        R5 → S10 = ([V, A], [R], [])     4
        R6 → S11 = ([V], [R, A], [])     3
        R3 → S4: poda
        ABERTOS  = [S5(3), S7(3), S8(3), S9(3), S11(3), S10(4)]
        FECHADOS = [S0(0), S1(1), S3(1), S4(1), S2(2), S6(2)]

    S5, S7 e S8 custam 3 como S9, mas foram gerados antes e saem primeiro.
        S5 gera S12(4).  S7 gera S13(5).  S8 gera S14(5).
        ABERTOS  = [S9(3), S11(3), S10(4), S12(4), S13(5), S14(5)]
        FECHADOS = [S0(0), S1(1), S3(1), S4(1), S2(2), S6(2), S5(3),
                    S7(3), S8(3)]

    Chega a vez de S9 = ([], [R, V], [A])
        ✓ SUCESSO

5. Árvore de busca (custo acumulado entre parênteses)

    S0 ([V,R], [A], [])  (0)
    ├── R1 → S1 ([V], [A,R], [])  (1)
    │   └── R2 → S5 ([], [A,R], [V])  (3)
    │       └── R3 → S12  (4)
    ├── R2 → S2 ([V], [A], [R])  (2)
    │   ├── R1 → S7 ([], [A,V], [R])  (3)
    │   │   └── R5 → S13  (5)
    │   └── R3 → S8 ([V,A], [], [R])  (3)
    │       └── R5 → S14  (5)
    ├── R3 → S3 ([V,R,A], [], [])  (1)   (impasse)
    └── R4 → S4 ([V,R], [], [A])  (1)
        └── R1 → S6 ([V], [R], [A])  (2)
            ├── R1 → S9 ([], [R,V], [A])  (3)   (objetivo)
            ├── R5 → S10 ([V,A], [R], [])  (4)
            └── R6 → S11 ([V], [R,A], [])  (3)

6. Caminho solução

    S0 ([V,R], [A], [])
     │ R4: azul H2 → H3        custo 1
     ↓
    S4 ([V,R], [], [A])
     │ R1: vermelho H1 → H2    custo 1
     ↓
    S6 ([V], [R], [A])
     │ R1: verde H1 → H2       custo 1
     ↓
    Sf ([], [R,V], [A])

    Custo da solução: 1 + 1 + 1 = 3
    Número de movimentos: 3      Iterações: 10      Gerados: 15

    Com as ordens decrescente e custom: o mesmo caminho em 11 iterações.

7. Conclusão

    Um nó só é expandido depois de todos os mais baratos, então quando o
    objetivo vira o estado atual nenhum nó aberto chega nele mais barato:
    a solução é a de MENOR CUSTO, que nem sempre é a de menos movimentos.
    Com custo unitário ela seria a busca em largura. Aqui, em P1, as duas
    acham o mesmo caminho, mas a ordenada deixa para depois os ramos que
    começam por R2 e chega em 10 iterações contra 14.
"""

from __future__ import annotations

from typing import ClassVar

from config import settings
from core.algorithms.domain.base import SearchAlgorithm
from core.domain.problem import Problem
from core.domain.state import State
from core.rules.domain.base import TransitionRule
from core.rules.strategies.domain.base import ControlStrategy
from core.search_tree.frontier import PriorityFrontier
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome
from core.search_tree.tree import SearchTree


class OrderedSearch(SearchAlgorithm):
    name: ClassVar[str] = "ordered"

    def __init__(
        self,
        tree: SearchTree,
        strategy: ControlStrategy,
        *,
        max_iterations: int = settings.MAX_ITERATIONS,
    ) -> None:
        super().__init__(tree, strategy, max_iterations=max_iterations)
        self._best_cost: dict[State, int] = {}
        self._open: dict[State, Node] = {}

    def _search(self, problem: Problem) -> Outcome:
        """Sempre expande o nó aberto de menor custo acumulado.

        O objetivo só encerra a busca quando vira o estado atual, não quando
        é gerado: só aí nenhum nó aberto pode chegar nele mais barato.
        """
        context = self._require_context()
        frontier = PriorityFrontier()
        frontier.push(context.root)

        # Vetor de menor custo: o custo do caminho mais barato já gerado até
        # cada estado. Cobre ABERTOS e FECHADOS de uma vez.
        self._best_cost = {context.root.state: 0}
        # O nó de cada estado que ainda espera em ABERTOS, para poder tirá-lo
        # de lá quando um caminho mais barato até o mesmo estado aparecer.
        self._open = {context.root.state: context.root}

        while len(frontier):
            if not self._next_iteration():
                return Outcome.CUTOFF

            node = frontier.pop()
            del self._open[node.state]
            if problem.is_goal(node.state):
                return self._succeed(node)

            self._visit(node)
            candidates = self._applicable_rules(node)
            if not candidates:
                self._deadlock(node)

            for rule in candidates:
                child = self._expand(node, rule)
                older = self._open.get(child.state)
                if older is not None:
                    # Chegou mais barato a um estado que ainda está aberto:
                    # o nó antigo sai de ABERTOS e da árvore.
                    frontier.remove(older)
                    self._discard(older)
                self._best_cost[child.state] = child.cost
                self._open[child.state] = child
                frontier.push(child)

        return self._exhausted()

    def _is_repetition(
        self, node: Node, rule: TransitionRule, successor: State
    ) -> bool:
        """Poda o filho que não melhora o menor custo conhecido do estado.

        Com custos positivos, um estado já fechado nunca é alcançado mais
        barato depois, então ele sempre cai aqui.
        """
        best = self._best_cost.get(successor)
        return best is not None and node.cost + rule.cost >= best

    def _discard(self, node: Node) -> None:
        """Só contabilidade: registra o nó substituído como poda do pai."""
        context = self._require_context()
        parent, rule = node.parent, node.rule
        assert parent is not None and rule is not None
        context.trace.record_prune(
            iteration=context.metrics.iterations,
            node=parent,
            rule_id=rule.id,
            state=node.state,
        )

## libs/inputs/

In [ ]:
%%writefile src/libs/inputs/exceptions.py
from __future__ import annotations


class InputError(Exception): ...


class InvalidArgumentError(InputError):
    def __init__(self, detail: str) -> None:
        super().__init__(f"invalid argument: {detail}")
        self.detail = detail


class UnknownSelectionError(InputError):
    def __init__(self, kind: str, name: str, available: tuple[str, ...]) -> None:
        super().__init__(f"unknown {kind}: {name!r}")
        self.kind = kind
        self.name = name
        self.available = available

In [ ]:
%%writefile src/libs/inputs/parser.py
from __future__ import annotations

import argparse
from collections.abc import Iterable, Sequence
from pathlib import Path
from typing import NoReturn

from config import settings
from core.algorithms.domain.registry import ALGORITHM_NAMES
from core.domain.problem import PROBLEM_IDS
from core.rules.strategies.custom_order import parse_order
from core.rules.strategies.domain.registry import STRATEGY_NAMES
from libs.inputs.exceptions import InvalidArgumentError
from libs.inputs.selection import ExecutionRequest


class _HelpFormatter(argparse.HelpFormatter):
    def add_usage(
        self,
        usage: str | None,
        actions: Iterable[argparse.Action],
        groups: Iterable[argparse._MutuallyExclusiveGroup],
        prefix: str | None = None,
    ) -> None:
        super().add_usage(usage, actions, groups, settings.USAGE_PREFIX)


class _Parser(argparse.ArgumentParser):
    def error(self, message: str) -> NoReturn:
        raise InvalidArgumentError(message)

    def exit(self, status: int = 0, message: str | None = None) -> NoReturn:
        raise InvalidArgumentError(message or "")


def build_parser() -> argparse.ArgumentParser:
    parser = _Parser(
        prog=settings.PROGRAM_NAME,
        description=settings.DESCRIPTION,
        epilog=settings.EPILOG,
        add_help=False,
        formatter_class=_HelpFormatter,
    )
    group = parser.add_argument_group(settings.OPTIONS_TITLE)
    group.add_argument("-h", "--help", action="store_true", help=settings.HELP_HELP)
    group.add_argument("--problem", action="append", help=settings.HELP_PROBLEM)
    group.add_argument("--algorithm", action="append", help=settings.HELP_ALGORITHM)
    group.add_argument("--strategy", action="append", help=settings.HELP_STRATEGY)
    group.add_argument("--all", action="store_true", help=settings.HELP_ALL)
    group.add_argument("--all-goals", action="store_true", help=settings.HELP_ALL_GOALS)
    group.add_argument("--order", help=settings.HELP_ORDER)
    group.add_argument("--show-tree", action="store_true", help=settings.HELP_SHOW_TREE)
    group.add_argument(
        "--show-trace", action="store_true", help=settings.HELP_SHOW_TRACE
    )
    group.add_argument(
        "--show-states", action="store_true", help=settings.HELP_SHOW_STATES
    )
    group.add_argument(
        "--format", default=settings.DEFAULT_OUTPUT_FORMAT, help=settings.HELP_FORMAT
    )
    group.add_argument("--output", help=settings.HELP_OUTPUT)
    group.add_argument("--svg-dir", help=settings.HELP_SVG_DIR)
    group.add_argument(
        "--rank-mode", default=settings.DEFAULT_RANK_MODE, help=settings.HELP_RANK_MODE
    )
    group.add_argument(
        "--max-iterations",
        default=str(settings.MAX_ITERATIONS),
        help=settings.HELP_MAX_ITERATIONS,
    )
    group.add_argument("--seed", help=settings.HELP_SEED)
    group.add_argument("--list", action="store_true", help=settings.HELP_LIST)
    group.add_argument("--verbose", action="store_true", help=settings.HELP_VERBOSE)
    return parser


def help_text() -> str:
    return build_parser().format_help()


def parse(argv: Sequence[str]) -> ExecutionRequest:
    namespace = build_parser().parse_args(list(argv))

    select_all: bool = namespace.all
    problems = _selected(namespace.problem, PROBLEM_IDS, select_all, upper=True)
    algorithms = _selected(namespace.algorithm, ALGORITHM_NAMES, select_all)
    strategies = _selected(namespace.strategy, STRATEGY_NAMES, select_all)
    order = parse_order(namespace.order) if namespace.order else None

    return ExecutionRequest(
        problem_ids=problems,
        algorithm_names=algorithms,
        strategy_names=strategies,
        custom_order=order,
        show_tree=namespace.show_tree,
        show_trace=namespace.show_trace,
        show_states=namespace.show_states,
        output_format=namespace.format.lower(),
        rank_mode=namespace.rank_mode.lower(),
        max_iterations=_positive_int("--max-iterations", namespace.max_iterations),
        output_path=Path(namespace.output) if namespace.output else None,
        svg_dir=Path(namespace.svg_dir) if namespace.svg_dir else None,
        list_only=namespace.list,
        show_help=namespace.help,
        verbose=namespace.verbose,
        all_goals=namespace.all_goals,
        seed=None
        if namespace.seed is None
        else _positive_int("--seed", namespace.seed),
    )


def _selected(
    raw: list[str] | None,
    available: tuple[str, ...],
    select_all: bool,
    *,
    upper: bool = False,
) -> tuple[str, ...]:
    if select_all or not raw:
        return available
    names = [
        token.strip().upper() if upper else token.strip().lower()
        for entry in raw
        for token in entry.split(",")
        if token.strip()
    ]
    return tuple(dict.fromkeys(names))


def _positive_int(flag: str, raw: str) -> int:
    try:
        value = int(raw)
    except ValueError:
        raise InvalidArgumentError(f"{flag}={raw!r}") from None
    if value <= 0:
        raise InvalidArgumentError(f"{flag}={raw!r}")
    return value

In [ ]:
%%writefile src/libs/inputs/selection.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

from config import settings


@dataclass(frozen=True, slots=True)
class ExecutionRequest:
    problem_ids: tuple[str, ...]
    algorithm_names: tuple[str, ...]
    strategy_names: tuple[str, ...]
    custom_order: tuple[str, ...] | None = None
    show_tree: bool = False
    show_trace: bool = False
    show_states: bool = False
    output_format: str = settings.DEFAULT_OUTPUT_FORMAT
    rank_mode: str = settings.DEFAULT_RANK_MODE
    max_iterations: int = settings.MAX_ITERATIONS
    output_path: Path | None = None
    svg_dir: Path | None = None
    list_only: bool = False
    show_help: bool = False
    verbose: bool = False
    seed: int | None = None
    all_goals: bool = False

    @property
    def execution_count(self) -> int:
        return (
            len(self.problem_ids) * len(self.algorithm_names) * len(self.strategy_names)
        )

In [ ]:
%%writefile src/libs/inputs/validator.py
from __future__ import annotations

from collections.abc import Iterable

from config import settings
from core.algorithms.domain.registry import ALGORITHM_NAMES
from core.domain.problem import PROBLEM_IDS
from core.rules.strategies.domain.registry import STRATEGY_NAMES
from libs.inputs.exceptions import UnknownSelectionError
from libs.inputs.selection import ExecutionRequest


def validate(request: ExecutionRequest) -> ExecutionRequest:
    _require_known("problem", request.problem_ids, PROBLEM_IDS)
    _require_known("algorithm", request.algorithm_names, ALGORITHM_NAMES)
    _require_known("strategy", request.strategy_names, STRATEGY_NAMES)
    _require_known("format", (request.output_format,), settings.OUTPUT_FORMATS)
    _require_known("rank_mode", (request.rank_mode,), settings.RANK_MODES)
    return request


def _require_known(
    kind: str, selected: Iterable[str], available: tuple[str, ...]
) -> None:
    known = set(available)
    for name in selected:
        if name not in known:
            raise UnknownSelectionError(kind, name, available)

## libs/outputs/

In [ ]:
%%writefile src/libs/outputs/console.py
from __future__ import annotations

import sys
import textwrap
from typing import TextIO

from core.search_tree.result import SearchResult
from libs.outputs import state_render, theme, trace_render, tree_render
from libs.outputs.formatter import ProblemReport, Report
from libs.outputs.trace_render import render_table
from libs.ranking.leaderboard import Leaderboard, Stat, Summary, SummaryRow


class ConsoleFormatter:
    name = "console"

    def render(self, report: Report) -> str:
        blocks = (
            []
            if report.summary_only
            else [self._render_problem(entry, report) for entry in report.problems]
        )
        if report.summary is not None and (
            report.summary_only or len(report.problems) > 1
        ):
            blocks.append(self._render_summary(report.summary))
        return "\n\n".join(block for block in blocks if block)

    def _render_summary(self, summary: Summary) -> str:
        sections = [render_section(theme.SUMMARY_HEADER, render_summary(summary))]
        if summary.highlights:
            sections.append(
                render_section(theme.HIGHLIGHTS_HEADER, render_highlights(summary))
            )
        return "\n\n".join(sections)

    def _render_problem(self, entry: ProblemReport, report: Report) -> str:
        sections = [
            f"{theme.PROBLEM_HEADER} {entry.problem_id}",
            "",
            f"{theme.INITIAL_STATE_HEADER}: {state_render.render_inline(entry.initial_state)}",
            f"{theme.GOAL_STATE_HEADER}: {state_render.render_inline(entry.goal_state)}",
        ]
        for result in entry.results:
            sections.append("")
            sections.append(render_result_box(result))
            sections.extend(_render_details(result, report))
        if entry.leaderboard is not None:
            sections.extend(
                [
                    "",
                    render_section(
                        theme.LEADERBOARD_HEADER, render_leaderboard(entry.leaderboard)
                    ),
                ]
            )
        return "\n".join(sections)


def render_result_box(result: SearchResult) -> str:
    metrics = result.metrics
    moves = (
        theme.ABSENT if result.solution_length is None else str(result.solution_length)
    )
    cost = theme.ABSENT if result.solution_cost is None else str(result.solution_cost)
    path = (
        theme.ARROW.join(result.applied_rules) if result.applied_rules else theme.ABSENT
    )
    rows = (
        (theme.REPORT_LABELS["outcome"], theme.outcome_label(result.outcome)),
        (theme.REPORT_LABELS["moves"], moves),
        (theme.REPORT_LABELS["cost"], cost),
        (theme.REPORT_LABELS["path"], path),
        (theme.REPORT_LABELS["iterations"], str(metrics.iterations)),
        (theme.REPORT_LABELS["nodes_generated"], str(metrics.nodes_generated)),
        (theme.REPORT_LABELS["nodes_visited"], str(metrics.nodes_visited)),
        (theme.REPORT_LABELS["rules_tested"], str(metrics.rules_tested)),
        (theme.REPORT_LABELS["backtracks"], str(metrics.backtracks)),
        (theme.REPORT_LABELS["deadlocks"], str(metrics.deadlocks)),
        (theme.REPORT_LABELS["max_depth"], str(metrics.max_depth)),
        (theme.REPORT_LABELS["elapsed"], f"{metrics.elapsed_ms:.3f}"),
    )
    lines = [line for label, value in rows for line in _wrap_row(label, value)]
    return render_box(result.label, tuple(lines))


def _wrap_row(label: str, value: str) -> list[str]:
    available = theme.MAX_BOX_WIDTH - 2 - theme.LABEL_WIDTH
    chunks = textwrap.wrap(value, width=available) or [""]
    head = f"{label.ljust(theme.LABEL_WIDTH)}{chunks[0]}"
    tail = [f"{''.ljust(theme.LABEL_WIDTH)}{chunk}" for chunk in chunks[1:]]
    return [head, *tail]


def render_leaderboard(leaderboard: Leaderboard) -> str:
    scored = leaderboard.mode == "score"
    headers = theme.LEADERBOARD_HEADERS if scored else theme.LEADERBOARD_HEADERS[:-1]
    rows = []
    for row in leaderboard.rows:
        result = row.result
        cells = [
            str(row.position),
            result.algorithm,
            result.strategy,
            theme.outcome_label(result.outcome),
            theme.ABSENT
            if result.solution_length is None
            else str(result.solution_length),
            str(result.metrics.iterations),
            str(result.metrics.nodes_generated),
        ]
        if scored:
            cells.append(theme.ABSENT if row.score is None else f"{row.score:.1f}")
        rows.append(tuple(cells))
    return render_table(headers, tuple(rows))


def render_summary(summary: Summary) -> str:
    rows = tuple(
        (
            row.algorithm,
            row.strategy,
            f"{row.successes}/{row.runs}",
            str(row.deadlocks),
            _render_stat(row.moves),
            _render_stat(row.cost),
            _render_stat(row.iterations),
            _render_stat(row.nodes_generated),
        )
        for row in summary.rows
    )
    table = render_table(theme.SUMMARY_HEADERS, rows)
    return f"{theme.SUMMARY_LEGEND}\n\n{table}"


def render_highlights(summary: Summary) -> str:
    strategies: dict[str, int] = {}
    for row in summary.rows:
        strategies[row.algorithm] = strategies.get(row.algorithm, 0) + 1
    rows = tuple(
        (
            theme.CRITERION_LABELS[highlight.criterion],
            _render_winners(highlight.by_mean, strategies),
            _render_winners(highlight.by_median, strategies),
        )
        for highlight in summary.highlights
    )
    table = render_table(theme.HIGHLIGHTS_HEADERS, rows)
    return f"{theme.HIGHLIGHTS_LEGEND}\n\n{table}"


def _render_winners(winners: tuple[SummaryRow, ...], strategies: dict[str, int]) -> str:
    grouped: dict[str, list[str]] = {}
    for row in winners:
        grouped.setdefault(row.algorithm, []).append(row.strategy)
    return "; ".join(
        f"{algorithm} {theme.ALL_STRATEGIES}"
        if len(names) == strategies[algorithm]
        else f"{algorithm} ({', '.join(names)})"
        for algorithm, names in grouped.items()
    )


def _render_stat(stat: Stat | None) -> str:
    if stat is None:
        return theme.ABSENT
    return f"{stat.mean:.2f} / {stat.median:g}"


def render_box(title: str, lines: tuple[str, ...]) -> str:
    content_width = max((len(line) for line in lines), default=0)
    width = min(
        max(content_width + 2, len(title) + 6, theme.MIN_BOX_WIDTH),
        max(theme.MAX_BOX_WIDTH, len(title) + 6, content_width + 2),
    )

    header = f"{theme.BOX_HORIZONTAL} {title} "
    top = (
        theme.BOX_TOP_LEFT
        + header
        + theme.BOX_HORIZONTAL * (width - len(header))
        + theme.BOX_TOP_RIGHT
    )
    body = [
        f"{theme.BOX_VERTICAL} {line.ljust(width - 2)} {theme.BOX_VERTICAL}"
        for line in lines
    ]
    bottom = (
        theme.BOX_BOTTOM_LEFT + theme.BOX_HORIZONTAL * width + theme.BOX_BOTTOM_RIGHT
    )
    return "\n".join([top, *body, bottom])


def render_section(title: str, body: str) -> str:
    return f"{title}\n{theme.BOX_HORIZONTAL * len(title)}\n{body}"


def _render_details(result: SearchResult, report: Report) -> list[str]:
    sections: list[str] = []
    if report.show_states and result.solution_path:
        drawings = "\n\n".join(
            state_render.render_pegs(node.state) for node in result.solution_path
        )
        sections.extend(["", render_section(theme.STATES_HEADER, drawings)])
    if report.show_tree:
        sections.extend(
            [
                "",
                render_section(
                    theme.TREE_HEADER, tree_render.render_tree(result.trace)
                ),
            ]
        )
    if report.show_trace:
        sections.extend(
            [
                "",
                render_section(
                    theme.TRACE_HEADER, trace_render.render_trace(result.trace)
                ),
            ]
        )
    return sections


def write(text: str, stream: TextIO | None = None) -> None:
    target = sys.stdout if stream is None else stream
    target.write(f"{text}\n")


def write_error(text: str) -> None:
    sys.stderr.write(f"{text}\n")

In [ ]:
%%writefile src/libs/outputs/dot_render.py
from __future__ import annotations

from core.search_tree.result import SearchResult
from core.search_tree.trace import TraceEvent, TraceStep
from libs.outputs import state_render, theme
from libs.outputs.tree_render import removed_edges


def render_dot(result: SearchResult) -> str:
    trace = result.trace
    root_steps = trace.of_event(TraceEvent.ROOT)
    if not root_steps:
        return ""

    goal_orders = {step.node_order for step in trace.of_event(TraceEvent.GOAL)}
    deadlock_orders = {step.node_order for step in trace.of_event(TraceEvent.DEADLOCK)}
    visited_orders = {step.node_order for step in trace.of_event(TraceEvent.VISIT)}
    solution_orders = {node.order for node in result.solution_path}
    removed = removed_edges(trace)

    lines = [
        "digraph search {",
        f'  label="{_title(result)}"; labelloc=t; fontname="{theme.GRAPH_FONT}";',
        f'  node [shape=box, style="rounded,filled", fontname="{theme.GRAPH_MONO_FONT}", fontsize=10];',
        f'  edge [fontname="{theme.GRAPH_FONT}", fontsize=9];',
        _node(root_steps[0], theme.GRAPH_ROOT_FILL, on_path=bool(solution_orders)),
    ]

    for step in trace.of_event(TraceEvent.GENERATE):
        if step.parent_order is None or (step.parent_order, step.rule_id) in removed:
            continue
        on_path = step.node_order in solution_orders
        if step.node_order in goal_orders:
            fill = theme.GRAPH_GOAL_FILL
        elif step.node_order in deadlock_orders:
            fill = theme.GRAPH_DEADLOCK_FILL
        elif step.node_order in visited_orders:
            fill = theme.GRAPH_VISITED_FILL
        else:
            fill = theme.GRAPH_GENERATED_FILL
        lines.append(_node(step, fill, on_path=on_path))
        lines.append(_edge(step, on_path=on_path))

    lines.extend(_pruned(step) for step in trace.of_event(TraceEvent.PRUNE))
    lines.append("}")
    return "\n".join(lines)


def _title(result: SearchResult) -> str:
    parts = [result.problem, result.label, theme.outcome_label(result.outcome)]
    if result.solution_length is not None:
        parts.append(f"{result.solution_length} {theme.GRAPH_MOVES_LABEL}")
    if result.solution_cost is not None:
        parts.append(f"{theme.GRAPH_COST_LABEL} {result.solution_cost}")
    return " · ".join(parts)


def _node(step: TraceStep, fill: str, *, on_path: bool) -> str:
    label = f"#{step.node_order}\\n{state_render.render_inline(step.state)}"
    width = theme.GRAPH_PATH_WIDTH if on_path else 1
    color = theme.GRAPH_PATH_COLOR if on_path else theme.GRAPH_NODE_COLOR
    return (
        f'  n{step.node_order} [label="{label}", fillcolor="{fill}", '
        f'color="{color}", penwidth={width}];'
    )


def _edge(step: TraceStep, *, on_path: bool) -> str:
    width = theme.GRAPH_PATH_WIDTH if on_path else 1
    color = theme.GRAPH_PATH_COLOR if on_path else theme.GRAPH_NODE_COLOR
    return (
        f'  n{step.parent_order} -> n{step.node_order} [label="{step.rule_id}", '
        f'color="{color}", fontcolor="{color}", penwidth={width}];'
    )


def _pruned(step: TraceStep) -> str:
    ghost = f"p{step.node_order}_{step.rule_id}"
    return (
        f'  {ghost} [label="{step.rule_id}", shape=plaintext, style=solid, '
        f'fontcolor="{theme.GRAPH_PRUNED_COLOR}", fontsize=8];\n'
        f"  n{step.node_order} -> {ghost} [style=dotted, "
        f'color="{theme.GRAPH_PRUNED_COLOR}", arrowhead=none];'
    )

In [ ]:
%%writefile src/libs/outputs/formatter.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Protocol

from core.domain.state import State
from core.search_tree.result import SearchResult
from libs.ranking.leaderboard import Leaderboard, Summary


@dataclass(frozen=True, slots=True)
class ProblemReport:
    problem_id: str
    initial_state: State
    goal_state: State
    results: tuple[SearchResult, ...]
    leaderboard: Leaderboard | None = None


@dataclass(frozen=True, slots=True)
class Report:
    problems: tuple[ProblemReport, ...]
    summary: Summary | None = None
    show_tree: bool = False
    show_trace: bool = False
    show_states: bool = False
    summary_only: bool = False


class Formatter(Protocol):
    def render(self, report: Report) -> str: ...

In [ ]:
%%writefile src/libs/outputs/graph_writer.py
from __future__ import annotations

import shutil
import subprocess
from pathlib import Path

from libs.outputs import dot_render
from libs.outputs.formatter import Report

DOT_EXECUTABLE = "dot"


def graphviz_available() -> bool:
    return shutil.which(DOT_EXECUTABLE) is not None


def write_graphs(report: Report, directory: Path) -> tuple[Path, ...]:
    render = graphviz_available()
    written: list[Path] = []
    for entry in report.problems:
        target = directory / entry.problem_id
        target.mkdir(parents=True, exist_ok=True)
        for result in entry.results:
            source = dot_render.render_dot(result)
            stem = target / f"{result.algorithm}_{result.strategy}"
            dot_path = stem.with_suffix(".dot")
            dot_path.write_text(source, encoding="utf-8")
            written.append(dot_path)
            if render:
                written.append(_render_svg(dot_path, stem.with_suffix(".svg")))
    return tuple(written)


def _render_svg(dot_path: Path, svg_path: Path) -> Path:
    subprocess.run(
        [DOT_EXECUTABLE, "-Tsvg", str(dot_path), "-o", str(svg_path)],
        check=True,
        capture_output=True,
    )
    return svg_path

In [ ]:
%%writefile src/libs/outputs/json_writer.py
from __future__ import annotations

import json
from pathlib import Path

from core.domain.state import Peg, State
from core.search_tree.result import SearchResult
from libs.outputs import theme
from libs.outputs.formatter import ProblemReport, Report
from libs.ranking.leaderboard import Leaderboard, Stat, Summary, SummaryRow

type JsonValue = (
    str | int | float | bool | list["JsonValue"] | dict[str, "JsonValue"] | None
)


class JsonFormatter:
    name = "json"

    def render(self, report: Report) -> str:
        payload: dict[str, JsonValue] = {
            "problemas": [_problem_to_json(entry) for entry in report.problems]
        }
        if report.summary is not None:
            payload["resumo"] = _summary_to_json(report.summary)
        return json.dumps(payload, ensure_ascii=False, indent=2)


def write_file(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def _problem_to_json(entry: ProblemReport) -> JsonValue:
    payload: dict[str, JsonValue] = {
        "id": entry.problem_id,
        "estado_inicial": _state_to_json(entry.initial_state),
        "objetivo": _state_to_json(entry.goal_state),
        "execucoes": [_result_to_json(result) for result in entry.results],
    }
    if entry.leaderboard is not None:
        payload["placar"] = _leaderboard_to_json(entry.leaderboard)
    return payload


def _leaderboard_to_json(leaderboard: Leaderboard) -> JsonValue:
    return {
        "modo": leaderboard.mode,
        "posicoes": [
            {
                "posicao": row.position,
                "algoritmo": row.result.algorithm,
                "estrategia": row.result.strategy,
                "desfecho": theme.outcome_label(row.result.outcome),
                "movimentos": row.result.solution_length,
                "escore": row.score,
            }
            for row in leaderboard.rows
        ],
    }


def _summary_to_json(summary: Summary) -> JsonValue:
    return {
        "linhas": [
            {
                "algoritmo": row.algorithm,
                "estrategia": row.strategy,
                "execucoes": row.runs,
                "sucessos": row.successes,
                "impasses": row.deadlocks,
                "movimentos": _stat_to_json(row.moves),
                "custo": _stat_to_json(row.cost),
                "iteracoes": _stat_to_json(row.iterations),
                "nos_gerados": _stat_to_json(row.nodes_generated),
            }
            for row in summary.rows
        ],
        "melhores": [
            {
                "criterio": highlight.criterion,
                "pela_media": [_row_label(row) for row in highlight.by_mean],
                "pela_mediana": [_row_label(row) for row in highlight.by_median],
            }
            for highlight in summary.highlights
        ],
    }


def _row_label(row: SummaryRow) -> JsonValue:
    return {"algoritmo": row.algorithm, "estrategia": row.strategy}


def _stat_to_json(stat: Stat | None) -> JsonValue:
    if stat is None:
        return None
    return {"media": round(stat.mean, 2), "mediana": stat.median}


def _result_to_json(result: SearchResult) -> JsonValue:
    metrics = result.metrics
    return {
        "algoritmo": result.algorithm,
        "estrategia": result.strategy,
        "desfecho": theme.outcome_label(result.outcome),
        "movimentos": result.solution_length,
        "custo": result.solution_cost,
        "caminho": list(result.applied_rules),
        "metricas": {
            "iteracoes": metrics.iterations,
            "nos_gerados": metrics.nodes_generated,
            "nos_visitados": metrics.nodes_visited,
            "regras_testadas": metrics.rules_tested,
            "retrocessos": metrics.backtracks,
            "impasses": metrics.deadlocks,
            "profundidade_maxima": metrics.max_depth,
            "tempo_ms": round(metrics.elapsed_ms, 3),
        },
        "estados": [_state_to_json(node.state) for node in result.solution_path],
    }


def _state_to_json(state: State) -> JsonValue:
    return {
        theme.PEG_NAMES[peg]: [theme.disk_name(disk) for disk in state[peg]]
        for peg in Peg
    }

In [ ]:
%%writefile src/libs/outputs/state_render.py
from __future__ import annotations

from core.domain.state import CAPACITIES, Peg, State
from libs.outputs import theme


def render_inline(state: State) -> str:
    parts = []
    for peg in Peg:
        stack = state[peg]
        content = (
            ",".join(theme.disk_symbol(disk) for disk in stack)
            if stack
            else theme.EMPTY_PEG
        )
        parts.append(f"{theme.PEG_NAMES[peg]}[{content}]")
    return " ".join(parts)


def render_pegs(state: State) -> str:
    height = max(CAPACITIES.values())
    columns = [_column(state, peg, height) for peg in Peg]
    width = max(len(cell) for column in columns for cell in column)

    lines = [
        "  ".join(column[level].center(width) for column in columns).rstrip()
        for level in range(height)
    ]
    footer = "  ".join(
        f"{theme.PEG_NAMES[peg]}({CAPACITIES[peg]})".center(width) for peg in Peg
    )
    lines.append(footer.rstrip())
    return "\n".join(lines)


def _column(state: State, peg: Peg, height: int) -> list[str]:
    stack = state[peg]
    cells = ["" for _ in range(height - len(stack))]
    cells.extend(f"[{theme.disk_name(disk)}]" for disk in reversed(stack))
    capacity = CAPACITIES[peg]
    for level in range(height):
        if not cells[level] and height - level <= capacity:
            cells[level] = "|"
    return cells

In [ ]:
%%writefile src/libs/outputs/theme.py
from __future__ import annotations

from collections.abc import Mapping
from typing import Final

from core.rules.domain.base import Disk, Peg
from core.search_tree.outcome import Outcome
from core.search_tree.trace import TraceEvent

DISK_NAMES: Final[Mapping[Disk, str]] = {
    Disk.GREEN: "VERDE",
    Disk.RED: "VERMELHO",
    Disk.BLUE: "AZUL",
}

DISK_SYMBOLS: Final[Mapping[Disk, str]] = {
    Disk.GREEN: "V",
    Disk.RED: "R",
    Disk.BLUE: "A",
}

PEG_NAMES: Final[Mapping[Peg, str]] = {
    Peg.H1: "H1",
    Peg.H2: "H2",
    Peg.H3: "H3",
}

OUTCOME_LABELS: Final[Mapping[Outcome, str]] = {
    Outcome.SUCCESS: "SUCESSO",
    Outcome.DEADLOCK: "IMPASSE",
    Outcome.CUTOFF: "LIMITE",
    Outcome.FAILURE: "FALHA",
}

EVENT_LABELS: Final[Mapping[TraceEvent, str]] = {
    TraceEvent.ROOT: "raiz",
    TraceEvent.VISIT: "visita",
    TraceEvent.GENERATE: "gera",
    TraceEvent.PRUNE: "poda",
    TraceEvent.GOAL: "objetivo",
    TraceEvent.DEADLOCK: "impasse",
    TraceEvent.BACKTRACK: "retrocesso",
    TraceEvent.CUTOFF: "limite",
    TraceEvent.EXHAUSTED: "esgotado",
}

REPORT_LABELS: Final[Mapping[str, str]] = {
    "outcome": "Desfecho",
    "moves": "Movimentos",
    "cost": "Custo",
    "path": "Caminho",
    "iterations": "Iterações",
    "nodes_generated": "Nós gerados",
    "nodes_visited": "Nós visitados",
    "rules_tested": "Regras testadas",
    "backtracks": "Retrocessos",
    "deadlocks": "Impasses",
    "max_depth": "Profundidade máx",
    "elapsed": "Tempo (ms)",
}

LEADERBOARD_HEADERS: Final[tuple[str, ...]] = (
    "#",
    "ALGORITMO",
    "ESTRATÉGIA",
    "DESFECHO",
    "MOV",
    "ITER",
    "GERADOS",
    "ESCORE",
)

SUMMARY_HEADERS: Final[tuple[str, ...]] = (
    "ALGORITMO",
    "ESTRATÉGIA",
    "SUCESSOS",
    "IMPASSES",
    "MOVIMENTOS",
    "CUSTO",
    "ITERAÇÕES",
    "GERADOS",
)

SUMMARY_LEGEND: Final = (
    "Cada métrica: média / mediana. Movimentos e custo contam só os sucessos."
)

HIGHLIGHTS_HEADERS: Final[tuple[str, ...]] = (
    "CRITÉRIO",
    "MELHOR PELA MÉDIA",
    "MELHOR PELA MEDIANA",
)

HIGHLIGHTS_LEGEND: Final = (
    "Menor valor vence. Só concorre quem tem o maior número de sucessos."
)

ALL_STRATEGIES: Final = "(todas)"

CRITERION_LABELS: Final[Mapping[str, str]] = {
    "moves": "Movimentos",
    "cost": "Custo",
    "iterations": "Iterações",
    "nodes_generated": "Nós gerados",
}

TREE_HEADER: Final = "Árvore de busca"
TRACE_HEADER: Final = "Passo a passo"
STATES_HEADER: Final = "Estados do caminho solução"
LEADERBOARD_HEADER: Final = "Placar"
SUMMARY_HEADER: Final = "Resumo consolidado"
HIGHLIGHTS_HEADER: Final = "Melhor por critério"
PROBLEM_HEADER: Final = "PROBLEMA"
INITIAL_STATE_HEADER: Final = "Posição inicial"
GOAL_STATE_HEADER: Final = "Objetivo"

ABSENT: Final = "—"
ARROW: Final = " → "
EMPTY_PEG: Final = "·"

BOX_TOP_LEFT: Final = "╭"
BOX_TOP_RIGHT: Final = "╮"
BOX_BOTTOM_LEFT: Final = "╰"
BOX_BOTTOM_RIGHT: Final = "╯"
BOX_HORIZONTAL: Final = "─"
BOX_VERTICAL: Final = "│"

TREE_BRANCH: Final = "├── "
TREE_LAST_BRANCH: Final = "└── "
TREE_TRUNK: Final = "│   "
TREE_GAP: Final = "    "

GRAPH_FONT: Final = "Helvetica"
GRAPH_MONO_FONT: Final = "monospace"
GRAPH_MOVES_LABEL: Final = "movimentos"
GRAPH_COST_LABEL: Final = "custo"
GRAPH_ROOT_FILL: Final = "#dbe9ff"
GRAPH_GOAL_FILL: Final = "#c6f0c6"
GRAPH_DEADLOCK_FILL: Final = "#f7c6c6"
GRAPH_VISITED_FILL: Final = "#ffffff"
GRAPH_GENERATED_FILL: Final = "#eeeeee"
GRAPH_NODE_COLOR: Final = "#333333"
GRAPH_PATH_COLOR: Final = "#1a56b8"
GRAPH_PRUNED_COLOR: Final = "#999999"
GRAPH_PATH_WIDTH: Final = 2.5

LABEL_WIDTH: Final = 18
MIN_BOX_WIDTH: Final = 56
MAX_BOX_WIDTH: Final = 76


def disk_symbol(disk: Disk) -> str:
    return DISK_SYMBOLS[disk]


def disk_name(disk: Disk) -> str:
    return DISK_NAMES[disk]


def outcome_label(outcome: Outcome) -> str:
    return OUTCOME_LABELS[outcome]


def event_label(event: TraceEvent) -> str:
    return EVENT_LABELS[event]

In [ ]:
%%writefile src/libs/outputs/trace_render.py
from __future__ import annotations

from core.search_tree.trace import Trace
from libs.outputs import state_render, theme

_HEADERS = ("ITER", "EVENTO", "NÓ", "PROF", "REGRA", "ESTADO")


def render_trace(trace: Trace) -> str:
    rows = [
        (
            str(step.iteration),
            theme.event_label(step.event),
            f"#{step.node_order}",
            str(step.depth),
            step.rule_id or theme.ABSENT,
            state_render.render_inline(step.state),
        )
        for step in trace
    ]
    return render_table(_HEADERS, tuple(rows))


def render_table(headers: tuple[str, ...], rows: tuple[tuple[str, ...], ...]) -> str:
    widths = [
        max(len(headers[column]), *(len(row[column]) for row in rows))
        if rows
        else len(headers[column])
        for column in range(len(headers))
    ]
    lines = [
        "  ".join(header.ljust(widths[i]) for i, header in enumerate(headers)).rstrip()
    ]
    lines.extend(
        "  ".join(cell.ljust(widths[i]) for i, cell in enumerate(row)).rstrip()
        for row in rows
    )
    return "\n".join(lines)

In [ ]:
%%writefile src/libs/outputs/tree_render.py
from __future__ import annotations

from core.search_tree.trace import Trace, TraceEvent
from libs.outputs import state_render, theme


def render_tree(trace: Trace) -> str:
    root_steps = trace.of_event(TraceEvent.ROOT)
    if not root_steps:
        return ""

    root = root_steps[0]
    removed = removed_edges(trace)
    children: dict[int, list[tuple[int, str | None, str]]] = {}
    for step in trace.of_event(TraceEvent.GENERATE):
        if step.parent_order is None or (step.parent_order, step.rule_id) in removed:
            continue
        label = state_render.render_inline(step.state)
        children.setdefault(step.parent_order, []).append(
            (step.node_order, step.rule_id, label)
        )

    goal_orders = {step.node_order for step in trace.of_event(TraceEvent.GOAL)}
    deadlock_orders = {step.node_order for step in trace.of_event(TraceEvent.DEADLOCK)}

    lines = [f"#{root.node_order} {state_render.render_inline(root.state)}"]
    _append_children(lines, root.node_order, children, goal_orders, deadlock_orders, "")
    return "\n".join(lines)


def _append_children(
    lines: list[str],
    parent_order: int,
    children: dict[int, list[tuple[int, str | None, str]]],
    goal_orders: set[int],
    deadlock_orders: set[int],
    prefix: str,
) -> None:
    siblings = children.get(parent_order, ())
    for position, (order, rule_id, label) in enumerate(siblings):
        is_last = position == len(siblings) - 1
        connector = theme.TREE_LAST_BRANCH if is_last else theme.TREE_BRANCH
        marks = ""
        if order in goal_orders:
            marks = f"  ({theme.event_label(TraceEvent.GOAL)})"
        elif order in deadlock_orders:
            marks = f"  ({theme.event_label(TraceEvent.DEADLOCK)})"
        lines.append(f"{prefix}{connector}{rule_id} → #{order} {label}{marks}")
        _append_children(
            lines,
            order,
            children,
            goal_orders,
            deadlock_orders,
            prefix + (theme.TREE_GAP if is_last else theme.TREE_TRUNK),
        )


def removed_edges(trace: Trace) -> set[tuple[int, str | None]]:
    return {
        (step.node_order, step.rule_id) for step in trace.of_event(TraceEvent.PRUNE)
    }

## libs/ranking/

In [ ]:
%%writefile src/libs/ranking/comparator.py
from __future__ import annotations

from collections.abc import Iterable

from core.search_tree.result import SearchResult
from libs.ranking import criteria


def lexicographic_key(result: SearchResult) -> tuple[int, int, int, int, str]:
    return (
        criteria.outcome_rank(result),
        criteria.solution_length(result),
        criteria.iterations(result),
        criteria.nodes_generated(result),
        criteria.label(result),
    )


def sort_results(results: Iterable[SearchResult]) -> tuple[SearchResult, ...]:
    return tuple(sorted(results, key=lexicographic_key))

In [ ]:
%%writefile src/libs/ranking/criteria.py
from __future__ import annotations

from collections.abc import Callable, Mapping
from dataclasses import dataclass
from typing import Final

from core.search_tree.outcome import Outcome
from core.search_tree.result import SearchResult

OUTCOME_ORDER: Final[Mapping[Outcome, int]] = {
    Outcome.SUCCESS: 0,
    Outcome.DEADLOCK: 1,
    Outcome.CUTOFF: 2,
    Outcome.FAILURE: 3,
}

NO_SOLUTION: Final = 10**6


def outcome_rank(result: SearchResult) -> int:
    return OUTCOME_ORDER[result.outcome]


def solution_length(result: SearchResult) -> int:
    length = result.solution_length
    return NO_SOLUTION if length is None else length


def iterations(result: SearchResult) -> int:
    return result.metrics.iterations


def nodes_generated(result: SearchResult) -> int:
    return result.metrics.nodes_generated


def label(result: SearchResult) -> str:
    return f"{result.algorithm}:{result.strategy}"


@dataclass(frozen=True, slots=True)
class Criterion:
    name: str
    measure: Callable[[SearchResult], int]
    weight: float


CRITERIA: Final[tuple[Criterion, ...]] = (
    Criterion("outcome", outcome_rank, 0.50),
    Criterion("solution_length", solution_length, 0.25),
    Criterion("iterations", iterations, 0.15),
    Criterion("nodes_generated", nodes_generated, 0.10),
)

In [ ]:
%%writefile src/libs/ranking/leaderboard.py
from __future__ import annotations

from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from statistics import fmean, median

from core.search_tree.outcome import Outcome
from core.search_tree.result import SearchResult
from libs.ranking import criteria
from libs.ranking.comparator import lexicographic_key, sort_results
from libs.ranking.scorer import score_all


@dataclass(frozen=True, slots=True)
class LeaderboardRow:
    position: int
    result: SearchResult
    score: float | None


@dataclass(frozen=True, slots=True)
class Leaderboard:
    problem_id: str
    mode: str
    rows: tuple[LeaderboardRow, ...]

    @property
    def winner(self) -> LeaderboardRow | None:
        return self.rows[0] if self.rows else None


@dataclass(frozen=True, slots=True)
class Stat:
    mean: float
    median: float


@dataclass(frozen=True, slots=True)
class SummaryRow:
    algorithm: str
    strategy: str
    runs: int
    successes: int
    deadlocks: int
    moves: Stat | None
    cost: Stat | None
    iterations: Stat
    nodes_generated: Stat


@dataclass(frozen=True, slots=True)
class Highlight:
    criterion: str
    by_mean: tuple[SummaryRow, ...]
    by_median: tuple[SummaryRow, ...]


@dataclass(frozen=True, slots=True)
class Summary:
    rows: tuple[SummaryRow, ...]
    highlights: tuple[Highlight, ...] = ()


SUMMARY_CRITERIA: tuple[tuple[str, Callable[[SummaryRow], Stat | None]], ...] = (
    ("moves", lambda row: row.moves),
    ("cost", lambda row: row.cost),
    ("iterations", lambda row: row.iterations),
    ("nodes_generated", lambda row: row.nodes_generated),
)


def build_leaderboard(
    problem_id: str, results: Sequence[SearchResult], mode: str
) -> Leaderboard:
    scores = score_all(results) if mode == "score" else {}
    ordered = (
        _sorted_by_score(results, scores) if mode == "score" else sort_results(results)
    )
    rows = tuple(
        LeaderboardRow(
            position=position,
            result=result,
            score=scores.get(criteria.label(result)),
        )
        for position, result in enumerate(ordered, start=1)
    )
    return Leaderboard(problem_id=problem_id, mode=mode, rows=rows)


def build_summary(results: Sequence[SearchResult]) -> Summary:
    grouped: dict[tuple[str, str], list[SearchResult]] = {}
    for result in results:
        grouped.setdefault((result.algorithm, result.strategy), []).append(result)

    rows = tuple(
        _summarise(algorithm, strategy, group)
        for (algorithm, strategy), group in grouped.items()
    )
    return Summary(
        rows=tuple(sorted(rows, key=_summary_key)), highlights=_highlights(rows)
    )


def _highlights(rows: Sequence[SummaryRow]) -> tuple[Highlight, ...]:
    if not rows:
        return ()
    most = max(row.successes for row in rows)
    contenders = [row for row in rows if row.successes == most]
    highlights: list[Highlight] = []
    for name, measure in SUMMARY_CRITERIA:
        measured: list[tuple[SummaryRow, Stat]] = []
        for row in contenders:
            stat = measure(row)
            if stat is not None:
                measured.append((row, stat))
        if not measured:
            continue
        best_mean = min(stat.mean for _, stat in measured)
        best_median = min(stat.median for _, stat in measured)
        highlights.append(
            Highlight(
                criterion=name,
                by_mean=tuple(r for r, s in measured if s.mean == best_mean),
                by_median=tuple(r for r, s in measured if s.median == best_median),
            )
        )
    return tuple(highlights)


def _stat(values: Sequence[int]) -> Stat | None:
    if not values:
        return None
    return Stat(mean=fmean(values), median=float(median(values)))


def _sorted_by_score(
    results: Sequence[SearchResult], scores: Mapping[str, float]
) -> tuple[SearchResult, ...]:
    return tuple(
        sorted(
            results,
            key=lambda result: (
                -scores[criteria.label(result)],
                lexicographic_key(result),
            ),
        )
    )


def _summarise(
    algorithm: str, strategy: str, group: Sequence[SearchResult]
) -> SummaryRow:
    moves = [r.solution_length for r in group if r.solution_length is not None]
    costs = [r.solution_cost for r in group if r.solution_cost is not None]
    iterations = _stat([result.metrics.iterations for result in group])
    generated = _stat([result.metrics.nodes_generated for result in group])
    assert iterations is not None and generated is not None
    return SummaryRow(
        algorithm=algorithm,
        strategy=strategy,
        runs=len(group),
        successes=sum(result.outcome.is_success for result in group),
        deadlocks=sum(result.outcome is Outcome.DEADLOCK for result in group),
        moves=_stat(moves),
        cost=_stat(costs),
        iterations=iterations,
        nodes_generated=generated,
    )


def _summary_key(row: SummaryRow) -> tuple[int, float, float, float, str, str]:
    no_solution = float(criteria.NO_SOLUTION)
    return (
        -row.successes,
        row.moves.mean if row.moves is not None else no_solution,
        row.moves.median if row.moves is not None else no_solution,
        row.iterations.mean,
        row.algorithm,
        row.strategy,
    )

In [ ]:
%%writefile src/libs/ranking/scorer.py
from __future__ import annotations

from collections.abc import Mapping, Sequence

from core.search_tree.result import SearchResult
from libs.ranking import criteria
from libs.ranking.criteria import Criterion

MAX_SCORE = 100.0


def score_all(results: Sequence[SearchResult]) -> Mapping[str, float]:
    if not results:
        return {}
    best = {
        criterion.name: min(criterion.measure(result) for result in results)
        for criterion in criteria.CRITERIA
    }
    return {criteria.label(result): _score(result, best) for result in results}


def _score(result: SearchResult, best: Mapping[str, int]) -> float:
    total = sum(
        criterion.weight * _normalise(criterion, result, best[criterion.name])
        for criterion in criteria.CRITERIA
    )
    return round(total * MAX_SCORE, 2)


def _normalise(criterion: Criterion, result: SearchResult, best: int) -> float:
    value = criterion.measure(result)
    if value == best:
        return 1.0
    if value <= 0:
        return 1.0
    return best / value if best > 0 else 1.0 / (1.0 + value)

## runner/

In [ ]:
%%writefile src/runner/cli.py
from __future__ import annotations

from collections.abc import Sequence
from pathlib import Path

from config import settings
from config.logging import configure_logging
from core.algorithms.domain.registry import ALGORITHM_NAMES
from core.domain.problem import PROBLEMS
from core.rules.domain.exceptions import InvalidRuleOrderError
from core.rules.strategies.domain.registry import STRATEGY_NAMES
from libs.inputs import parser, validator
from libs.inputs.exceptions import (
    InputError,
    InvalidArgumentError,
    UnknownSelectionError,
)
from libs.inputs.selection import ExecutionRequest
from libs.outputs import console, graph_writer, state_render
from libs.outputs.console import ConsoleFormatter
from libs.outputs.formatter import Formatter, Report
from libs.outputs.json_writer import JsonFormatter, write_file
from runner.pipeline import build_report

_FORMATTERS: dict[str, Formatter] = {
    ConsoleFormatter.name: ConsoleFormatter(),
    JsonFormatter.name: JsonFormatter(),
}


def main(argv: Sequence[str] | None = None) -> int:
    arguments = [] if argv is None else list(argv)
    try:
        request = validator.validate(parser.parse(arguments))
    except (InputError, InvalidRuleOrderError) as error:
        console.write_error(_error_message(error))
        console.write_error(settings.ERROR_HINT)
        return settings.EXIT_INVALID_INPUT

    configure_logging(verbose=request.verbose)

    if request.show_help:
        console.write(parser.help_text())
        return settings.EXIT_OK

    if request.list_only:
        console.write(_render_catalogues())
        return settings.EXIT_OK

    report = build_report(request)
    _emit(report, request)
    if request.svg_dir is not None:
        _write_graphs(report, request.svg_dir)
    return settings.EXIT_OK


def _write_graphs(report: Report, directory: Path) -> None:
    written = graph_writer.write_graphs(report, directory)
    console.write(settings.GRAPHS_WRITTEN.format(count=len(written), path=directory))
    if not graph_writer.graphviz_available():
        console.write_error(settings.GRAPHVIZ_MISSING)


def _emit(report: Report, request: ExecutionRequest) -> None:
    text = _FORMATTERS[request.output_format].render(report)
    if request.output_path is None:
        console.write(text)
        return
    write_file(request.output_path, text)
    console.write(settings.OUTPUT_WRITTEN.format(path=request.output_path))


def _render_catalogues() -> str:
    problems = "\n".join(
        f"  {problem.id}  {state_render.render_inline(problem.goal)}"
        for problem in PROBLEMS
    )
    return "\n\n".join(
        [
            console.render_section(settings.LIST_PROBLEMS_HEADER, problems),
            console.render_section(
                settings.LIST_ALGORITHMS_HEADER,
                "\n".join(f"  {name}" for name in ALGORITHM_NAMES),
            ),
            console.render_section(
                settings.LIST_STRATEGIES_HEADER,
                "\n".join(f"  {name}" for name in STRATEGY_NAMES),
            ),
        ]
    )


def _error_message(error: Exception) -> str:
    match error:
        case UnknownSelectionError():
            detail = settings.ERROR_UNKNOWN_SELECTION.format(
                kind=settings.KIND_LABELS[error.kind],
                name=error.name,
                available=", ".join(error.available),
            )
        case InvalidRuleOrderError():
            detail = settings.ERROR_INVALID_RULE_ORDER.format(reason=error.reason)
        case InvalidArgumentError():
            detail = settings.ERROR_INVALID_ARGUMENT.format(detail=error.detail)
        case _:
            detail = str(error)
    return f"{settings.ERROR_PREFIX} {detail}"

In [ ]:
%%writefile src/runner/executor.py
from __future__ import annotations

from core.algorithms.domain.registry import get_algorithm
from core.domain.problem import Problem, all_goal_problems, get_problem
from core.rules.strategies.custom_order import CUSTOM, CustomOrderStrategy
from core.rules.strategies.domain.base import ControlStrategy
from core.rules.strategies.domain.registry import get_strategy
from core.search_tree.result import SearchResult
from core.search_tree.tree import SearchTree
from libs.inputs.selection import ExecutionRequest


def run_problem(
    problem: Problem, request: ExecutionRequest
) -> tuple[SearchResult, ...]:
    return tuple(
        get_algorithm(algorithm_name)(
            SearchTree(),
            resolve_strategy(strategy_name, request.custom_order),
            max_iterations=request.max_iterations,
        ).solve(problem)
        for algorithm_name in request.algorithm_names
        for strategy_name in request.strategy_names
    )


def run_matrix(
    request: ExecutionRequest,
) -> tuple[tuple[Problem, tuple[SearchResult, ...]], ...]:
    problems = (
        all_goal_problems()
        if request.all_goals
        else tuple(get_problem(problem_id) for problem_id in request.problem_ids)
    )
    return tuple((problem, run_problem(problem, request)) for problem in problems)


def resolve_strategy(
    name: str, custom_order: tuple[str, ...] | None
) -> ControlStrategy:
    if name == CUSTOM.name and custom_order is not None:
        return CustomOrderStrategy(sequence=custom_order)
    return get_strategy(name)

In [ ]:
%%writefile src/runner/pipeline.py
from __future__ import annotations

from libs.inputs.selection import ExecutionRequest
from libs.outputs.formatter import ProblemReport, Report
from libs.ranking.leaderboard import build_leaderboard, build_summary
from runner.executor import run_matrix


def build_report(request: ExecutionRequest) -> Report:
    executed = run_matrix(request)
    problems = tuple(
        ProblemReport(
            problem_id=problem.id,
            initial_state=problem.initial,
            goal_state=problem.goal,
            results=results,
            leaderboard=build_leaderboard(problem.id, results, request.rank_mode),
        )
        for problem, results in executed
    )
    every_result = [result for _, results in executed for result in results]
    return Report(
        problems=problems,
        summary=build_summary(every_result),
        show_tree=request.show_tree,
        show_trace=request.show_trace,
        show_states=request.show_states,
        summary_only=request.all_goals,
    )

## Apresentação

In [ ]:
#@title Funções de apresentação
import shutil
import subprocess

import pandas as pd
from IPython.display import SVG, display

from libs.outputs import dot_render, state_render, theme, trace_render, tree_render

LABELS = theme.REPORT_LABELS


def metrics_table(results):
    rows = []
    for result in results:
        metrics = result.metrics
        rows.append(
            {
                "Estratégia": result.strategy,
                LABELS["outcome"]: theme.outcome_label(result.outcome),
                LABELS["moves"]: result.solution_length,
                LABELS["cost"]: result.solution_cost,
                LABELS["path"]: " → ".join(result.applied_rules) or theme.ABSENT,
                LABELS["iterations"]: metrics.iterations,
                LABELS["nodes_generated"]: metrics.nodes_generated,
                LABELS["nodes_visited"]: metrics.nodes_visited,
                LABELS["backtracks"]: metrics.backtracks,
                LABELS["deadlocks"]: metrics.deadlocks,
                LABELS["max_depth"]: metrics.max_depth,
            }
        )
    return (
        pd.DataFrame(rows)
        .set_index("Estratégia")
        .astype({LABELS["moves"]: "Int64", LABELS["cost"]: "Int64"})
    )


def show_path(result):
    for step, node in enumerate(result.solution_path):
        rule = "início" if node.rule is None else f"{node.rule.id}: {node.rule.origin.name} → {node.rule.destination.name}"
        print(f"passo {step}  {rule}  (custo acumulado {node.cost})")
        print(state_render.render_pegs(node.state))
        print()


def show_tree(result):
    source = dot_render.render_dot(result)
    if shutil.which("dot") is None:
        print(tree_render.render_tree(result.trace))
        return
    svg = subprocess.run(
        ["dot", "-Tsvg"], input=source, capture_output=True, text=True, check=True
    ).stdout
    display(SVG(svg))


def show_trace(result):
    print(trace_render.render_trace(result.trace))

# Busca irrevogável

Um caminho só. A regra preferida da estratégia é aplicada e as outras são esquecidas: cada passo é definitivo e não há retorno. Uma regra que levaria a um estado já percorrido é podada.

Sempre para, no máximo depois dos 36 estados, porque nunca repete estado no caminho. Parar não é chegar: é o único método em que a estratégia muda o desfecho.

## Fluxograma

In [ ]:
#@title Desenha o fluxograma
FLOW = '''digraph flow {
  rankdir=TB; fontname="Helvetica"; bgcolor="transparent";
  node [fontname="Helvetica", fontsize=11, shape=box, style="rounded,filled", fillcolor="#ffffff", color="#333333"];
  edge [fontname="Helvetica", fontsize=10, color="#333333"];
  root [label="raiz", shape=oval, fillcolor="#dbe9ff"];
  goal [label="é o objetivo?", shape=diamond];
  success [label="SUCESSO", shape=oval, fillcolor="#c6f0c6"];
  visit [label="visita: regras aplicáveis na ordem da estratégia,\nmenos as que repetem estado do caminho"];
  left [label="sobrou regra?", shape=diamond];
  deadlock [label="IMPASSE", shape=oval, fillcolor="#f7c6c6"];
  apply [label="aplica só a primeira e gera o filho"];
  root -> goal;
  goal -> success [label="sim"];
  goal -> visit [label="não"];
  visit -> left;
  left -> deadlock [label="não"];
  left -> apply [label="sim"];
  apply -> goal;
}'''
if shutil.which("dot"):
    display(SVG(subprocess.run(["dot", "-Tsvg"], input=FLOW, capture_output=True, text=True, check=True).stdout))
else:
    print(FLOW)

## O código

O laço inteiro cabe em poucas linhas: testa o objetivo, visita, pega as regras permitidas e aplica a primeira. Sem regra, é impasse e a busca acaba. O docstring do módulo traz a execução de P1 resolvida à mão, com a ordem decrescente.

In [ ]:
%%writefile src/core/algorithms/irrevocable.py
"""BUSCA IRREVOGÁVEL

1. Estado inicial e objetivo (carta P1)

    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])

       H1      H2      H3              H1      H2      H3
      ┌───┐   ┌───┐   ┌───┐           ┌───┐   ┌───┐   ┌───┐
      │ R │   │   │   │   │           │   │   │ V │   │   │
      │ V │   │ A │   │   │           │   │   │ R │   │ A │
      └───┘   └───┘   └───┘           └───┘   └───┘   └───┘

2. Regras válidas em S0

    R1 = H1 → H2  ✓          R4 = H2 → H3  ✓
    R2 = H1 → H3  ✓          R5 = H3 → H1  ✗  H3 vazia
    R3 = H2 → H1  ✓          R6 = H3 → H2  ✗  H3 vazia

3. Critério

    A estratégia ordena as regras válidas e a busca aplica a primeira.
    Não guarda as outras: cada passo é definitivo e não há retorno.
    Uma regra que leva a um estado já percorrido é descartada (poda).

4. Execução com a ordem decrescente R6 → R5 → R4 → R3 → R2 → R1

    Passo 1   S0 = ([V, R], [A], [])
              válidas em ordem: R4, R3, R2, R1
              aplica R4 (azul: H2 → H3)
              ↓
    Passo 2   S1 = ([V, R], [], [A])
              válidas em ordem: R6, R5, R1
              R6 volta a S0: poda.  Aplica R5 (azul: H3 → H1)
              ↓
    Passo 3   S2 = ([V, R, A], [], [])
              válidas em ordem: R2, R1
              R2 volta a S1: poda.  R1 volta a S0: poda.
              Nenhuma regra sobrou.
              ↓
              ✗ IMPASSE

5. Árvore de busca

    S0 ([V,R], [A], [])
    └── R4 → S1 ([V,R], [], [A])
        └── R5 → S2 ([V,R,A], [], [])   (impasse)

6. A mesma carta com a ordem crescente R1 → R2 → R3 → R4 → R5 → R6

    S0 ([V,R],[A],[]) → R1 → R2 → R3 → R3 → R5 → R1 → R1 → R2 → R3 → R3
       → R5 → R1 → R1 → R2 → Sf ([],[R,V],[A])

    ✓ SUCESSO com 14 movimentos.  O caminho ótimo tem 3.

7. Conclusão

    O caminho nunca repete estado, então a busca sempre para, no máximo
    depois dos 36 estados. Parar não é chegar: a ordem decrescente termina
    em impasse e a crescente só chega depois de 14 movimentos.
    É o único método em que a estratégia decide o desfecho.
"""

from __future__ import annotations

from typing import ClassVar

from core.algorithms.domain.base import SearchAlgorithm
from core.domain.problem import Problem
from core.search_tree.outcome import Outcome


class IrrevocableSearch(SearchAlgorithm):
    name: ClassVar[str] = "irrevocable"

    def _search(self, problem: Problem) -> Outcome:
        node = self._require_context().root
        while True:
            if not self._next_iteration():
                return Outcome.CUTOFF
            if problem.is_goal(node.state):
                return self._succeed(node)

            self._visit(node)
            candidates = self._applicable_rules(node)
            if not candidates:
                self._deadlock(node)
                return Outcome.DEADLOCK

            node = self._expand(node, candidates[0])

## Execução em P1

Uma execução por estratégia de controle, sobre a carta P1. O motor conta as métricas; a tabela só as organiza.

In [ ]:
from core.algorithms.domain.registry import get_algorithm
from core.domain.problem import get_problem
from core.rules.strategies.domain.registry import STRATEGIES
from core.search_tree.tree import SearchTree

problem = get_problem("P1")
Algorithm = get_algorithm("irrevocable")
results = {
    name: Algorithm(SearchTree(), strategy).solve(problem)
    for name, strategy in STRATEGIES.items()
}
metrics_table(results.values())

## Passo a passo

O trace da execução com a estratégia `descending`: cada evento do motor (visita, gera, poda, impasse, retrocesso, objetivo), com a iteração, o nó, a profundidade, a regra e o estado. Legenda dos discos: V verde, R vermelho, A azul.

In [ ]:
show_trace(results["descending"])

## Árvores de busca

Uma árvore por estratégia. O caminho solução está em azul, o objetivo em verde, os impasses em vermelho, os nós gerados e não visitados em cinza e as regras podadas em pontilhado.

In [ ]:
for result in results.values():
    show_tree(result)

## Caminho solução

O caminho da execução com a estratégia `ascending`, estado por estado.

In [ ]:
show_path(results["ascending"])

## Experimente

Troque a ordem das regras ou a carta e rode de novo. `all_goal_problems()` traz os 36 objetivos possíveis (G01 a G36).

In [ ]:
from core.domain.problem import all_goal_problems
from core.rules.strategies.custom_order import CustomOrderStrategy

order = ("R1", "R4", "R2", "R6", "R3", "R5")
target = all_goal_problems()[20]
result = Algorithm(SearchTree(), CustomOrderStrategy(sequence=order)).solve(target)
print(target.id, state_render.render_inline(target.goal))
print(theme.outcome_label(result.outcome), " → ".join(result.applied_rules) or theme.ABSENT)
show_tree(result)

## Conclusão

- Em P1, `descending` trava na 3ª iteração: R4 e R5 levam a `H1[V,R,A]`, e de lá as duas regras aplicáveis voltam a estados do caminho. `custom` também termina em impasse.
- `ascending` chega, mas com 14 movimentos, quando o ótimo tem 3.
- A busca é barata (no máximo um nó gerado por iteração), mas não garante solução nem qualidade. A estratégia decide tudo.